In [12]:
#%load_ext autoreload
#%autoreload 2

 
import sys, pathlib, json, pprint, pandas as pd 
from pathlib import Path
from pydantic import Field,BaseModel 
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from runtime.v4.semantics.load_semantics import load_semantics, load_idiom_rules
from runtime.v4.semantics.semantic_models import * 
from runtime.v4.analyst_agent.catalog import Catalog
from runtime.v4.analyst_agent.smart_data import SmartData
#from runtime.v4.analyst_agent.smart_data_tools  import SmartDataTools
from runtime.v4.analyst_agent.analyst_models  import *
from runtime.v4.analyst_agent.analyst_prompts import system_prompt_template3 as agent_system_prompt


import  os  
from datetime import datetime 
from langchain_core.tools import StructuredTool, Tool


import duckdb
from typing import Any, Dict, List, Iterable, Union, Optional 
import yaml
import pprint
import pandas as pd, numpy as np
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig, Runnable, RunnableLambda
 
from langchain.tools import tool, ToolRuntime
from langgraph.runtime import get_runtime 
from langchain.agents import create_agent
from langchain_core.runnables import RunnableLambda
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_core.runnables import Runnable
from langchain.tools import tool, ToolRuntime
from langgraph.runtime import get_runtime 
from langchain.agents import create_agent
 
from golden_queries import golden_injector_queries

In [13]:

#inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
#pinj = pd.read_csv("../datasets/IX5I_4P/producers.csv")
#locs= pd.read_csv("../datasets/IX5I_4P/locations.csv")

def fetch_data(path:str):
    inj  = pd.read_csv(path+"injectors.csv")
    pinj = pd.read_csv(path+"producers.csv")
    locs = pd.read_csv(path+"locations.csv")
    inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
    inj['DAY']   = inj['DATE'].dt.day
    inj['MONTH'] = inj['DATE'].dt.month
    inj['YEAR']  = inj['DATE'].dt.year
    pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
    pinj['DAY']   = pinj['DATE'].dt.day
    pinj['MONTH'] = pinj['DATE'].dt.month
    pinj['YEAR']  = pinj['DATE'].dt.year


    return inj, pinj, locs

path =  "../datasets/Demo1/"
inj, pinj, locs = fetch_data(path)


from get_llm_model import azure_llm_if
from support import * 

## Load semantics constrains and idioms

In [14]:
semantics_json = Path('../runtime/v4/semantics/semantic_models.json')
idiom_rules_path = Path('../runtime/v4/semantics/idioms.json')

semantic_catalog = load_semantics(semantics_json)
idiom_rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = idiom_rules_path )

constraints = "".join([f"- {i}\n" for i in semantic_catalog.semantic_constraints])
idiom_context = "".join([f"- {i}: {v}\n" for i,v in idiom_rules.items()])
known_table_models = { t.name: t for t in semantic_catalog.tables } 
df_dict = {'injectors': inj, 'producers':pinj , 'locations': locs }

## Create SmartData and Tools

In [15]:

smart_data = SmartData.initialize_from_named_dataframes( df_dict, known_table_models)
smart_data_tools =  smart_data.get_table_tools()#SmartDataTools( data=data).get_tools()
agent_tools = smart_data.get_tools() 


## Build analyst prompt

In [16]:

from runtime.v4.analyst_agent.analyst_prompts import system_prompt_template1b as system_prompt_template1
prompt = system_prompt_template1.format(idiom = 'duckdb', 
                                        idiom_examples = idiom_context, 
                                        constraints = constraints)

## Simple direct ReAct agent. 

In [17]:
llm = azure_llm_if()

system_prompt_template1c = """
You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database


===============================================================================
Workflow:
===============================================================================
You must:
1. Always call catalog_snapshot as the first step of the workflow.
After catalog_snapshot, check whether all requested concepts map clearly to catalog tables, columns, or known metrics.
If not, ask clarification and do not call sql_* tools.

2. Analyze the question and the information in the catalog and produce a concise PLAN
The PLAN must be concise and must include:
- required source tables
- whether existing derived tables can be reused
- target table names to materialize
- high-level transformation logic, without SQL


3. You MUST ALWAYS record the PLAN in plain text. Only after the PLAN message is sent may you call sql_* tools.
4. Use sql_materialize to create intermediate tables.
5. When multiple output tables are to be produced, proceed sequentially one at a time  
6. Your job finishes once all the target tables are confirmed present (either via initial audit or your materializations).  

===============================================================================
Important:
===============================================================================
- The name of generated tables and columns should reflect the table contents  

- Use lowercase snake_case for table names and column names 
    Example 1: yearly_aggregated_oil_producer_per_subzone
    Example 2: gas_oil_water_cummulated_volumes 

- Be explicit in the detailed description of tables produced 

- Sequential Execution:  If you need to materialize multiple tables, do so one by one, verifying the metadata for each.

===============================================================================
Output
===============================================================================
In each turn you will provide as result any of these:
1. Either a textual summary if the results the produced tables have less than 5 rows and less than 3 columns
2. A textual response asking for clarification if the question is ambiguous and cannot be answered with the available data
3. One or more tables if the results contain 5 or more rows and 3 or more columns 

Do not proceed if the question cannot be answered with the available data. 
Instead, ask for clarification 

===============================================================================
SQL generation rules 
===============================================================================
- ALWAYS use **{idiom}** compliant SQL syntax when generating queries.
{idiom_examples}

===============================================================================
Domain constraints
===============================================================================
{constraints}

===============================================================================
Chart-ready output rules
===============================================================================
CHART-READY OUTPUT RULES

For chart/plot/graph requests:

- "plot A by B"
  => return one row per B

- "plot A by B,C"
  => return one row per (B,C)

- "plot A by B,C,D"
  => return one row per (B,C,D)

Rules:
- Preserve all grouping columns.
- Aggregate A at the requested grouping level.
- Use sum by default for additive quantities unless another aggregation is requested.
- If multiple grouping columns together naturally define the chart axis, also create a readable display label column.
- Do not return raw detail rows for grouped chart requests.

Do not return raw detail rows when the user asks for aggregated chart-ready output.

===============================================================================
MANDATORY REUSE RULE:
===============================================================================

After calling catalog_snapshot:

1. If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.

2. You MUST NOT recompute intermediate tables if an equivalent derived table already exists.

3. Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.


Important:
- If a query depends on a table, ensure it has been materialized first.
- Always ask for clarification if the question is ambiguous or cannot be answered with the available data.
"""



In [18]:
 
#from runtime.v4.analyst_agent.analyst_prompts import system_prompt_template1c as system_prompt_template1

prompt = system_prompt_template1c.format(idiom = 'duckdb', 
                                        idiom_examples = idiom_context, 
                                        constraints = constraints)

smart_data.clear_derived()
agent = create_agent(
        model=llm,
        system_prompt=prompt,
        tools=agent_tools,
        response_format=AgentTableResponse,
        #checkpointer= MemorySaver() 
    )


#name = response['structured_response'].tables[0].table_name
#data.get_table_as_df( name )

In [27]:

mixed_question = "How many wells are there? and show producers ranked by cummulated oil production the last year" #'plot VQRp3 index'

query_question = "How many wells are there? Show the Projectivity-X3 computed via the kazanna method" 

tabular_question ="rank producers by cummulated oil production the last year and show the 3 of highest ranking" 

diagnostic_question = """Plot the Water-Oil Ratio WOR) and its derivative with " \
respect to cumulative oil production on a log-log scale for each producer in a different trace"""

query = diagnostic_question
messages = {"messages": [{"role": "user", "content": query}]}

response = run_agent_stream_values(agent, messages )

#response = agent.invoke(messages, config={"recursion_limit": 20})



--- HumanMessage ---
name: None
Plot the Water-Oil Ratio WOR) and its derivative with " respect to cumulative oil production on a log-log scale for each producer in a different trace

--- AIMessage ---
name: None
TOOL CALL: catalog_snapshot
ARGS: [skipped]

--- skipped tool output: catalog_snapshot ---

--- AIMessage ---
name: None
TOOL CALL: reuse_derived_table
ARGS: {'derived_table_name': 'water_oil_ratio_and_derivative', 'table_description': 'This table contains the Water-Oil Ratio (WOR) and its derivative with respect to cumulative oil production for each producer well over time. Columns include:\n- NAME: Producer well identifier\n- DATE: Production date\n- cumulative_oil: Cumulative oil production up to the given date\n- cumulative_water: Cumulative water production up to the given date\n- wor: Water-Oil Ratio (cumulative_water / cumulative_oil)\n- dwor_dcumoil: Derivative of WOR with respect to cumulative oil production'}

--- ToolMessage ---
name: reuse_derived_table
Table wate

In [28]:
response['structured_response']
user_query = response['structured_response'].user_query
table_name = response['structured_response'].tables[0].table_name
table = smart_data.catalog_snapshot( table_name ).derived_tables[0]
df = smart_data.get_table_as_df(table_name)
df


,NAME,DATE,cumulative_oil,cumulative_water,wor,dwor_dcumoil
0,BG-1494P,2020-10-05,0.000000,0.000000,NaN,NaN
1,BG-1494P,2020-10-06,0.000000,0.000000,NaN,NaN
2,BG-1494P,2020-10-07,335.559866,-0.024536,-0.000073,1.900044e-07
3,BG-1494P,2020-10-08,755.423290,0.005029,0.000007,9.003047e-08
4,BG-1494P,2020-10-09,1250.179862,0.064010,0.000051,-1.604745e-07
...,...,...,...,...,...,...
13756,BG-1502P,2021-11-21,244260.140810,268655.264808,1.099874,6.207604e-06
13757,BG-1502P,2021-11-22,244579.693631,269491.894799,1.101857,6.250027e-06
13758,BG-1502P,2021-11-23,244897.650338,270328.907578,1.103844,6.221732e-06
13759,BG-1502P,2021-11-24,245211.875387,271155.158243,1.105799,6.148902e-06


In [ ]:
raise "xcxx"

: 

## Example query 

#### Latency 

In [ ]:
from langchain_core.messages import HumanMessage

messages = [
    HumanMessage(content="list 10 countries in Asia and their capitals")
]
result = llm.invoke(messages, config={"recursion_limit": 20})

result 

: 

#### Golden queries

In [ ]:
query3_1 = "Tell me the total water injection volume for each quater since 2018 split by subzone and year"
query3_2 = "Tell me the maximum quarterly water injection volume for each year and subzone since 2018"
query3_3 = "Tell me the total water injection volume by quarter and subzone aggregated across all years since 2018"

x = """For each SUBZONE compute the year-over-year percentage change in total injection 
volume and report the largest drop"""

query_pie_1 = "Show the share of total water injection volume by subzone in the last year of the data"
query_pie_2 = "Show the percentage contribution of each quarter to total water injection volume since 2018"

time_series = "plot the water injected over time for well named BG-0718I"




: 

In [ ]:
messages = {"messages": [{"role": "user", "content": query3_1}]}

#response = run_agent_stream_values(agent, messages )

response = agent.invoke(messages, config={"recursion_limit": 20})

: 

In [ ]:
response['structured_response']

: 

In [ ]:

user_query = response['structured_response'].user_query
table_name = response['structured_response'].tables[0].table_name
table = smart_data.catalog_snapshot( table_name ).derived_tables[0]
df = smart_data.get_table_as_df(table_name)
df

: 

#### Golden queries for injectors only 



In [ ]:
for query in golden_injector_queries[9:10]:
    print(100*'=')
    print(f"QUERY: {query}")
    messages = {"messages": [{"role": "user", "content": query}]}
    response = run_agent_stream_values(agent, messages )
    
    if len(response['structured_response'].tables) > 0:
        #print("USER QUERY INTERPRETED BY AGENT:")
        #print(response['structured_response'].user_query)
        table_name = response['structured_response'].tables[0].table_name
        df = single_agent_data.get_table_as_df(table_name)
        print("RESULTING TABLE:")
        print(df)

    elif response['structured_response'].text is not None:
        print("USER QUERY INTERPRETED BY AGENT:")
        print(response['structured_response'].user_query)
        print("RESULTING TEXT:")
        print(response['structured_response'].text)

    else:
        print("No tables or text produced by the agent.")
           

: 

## The presentation component

The output from the previous step is always one or more tables.
A table can be the final presentation but in most cases, we should produce 
a chart. Tables are reserved for cases when the user specifically selected a 
table or the tables are small (2 rows or so)



In [29]:
df

,NAME,DATE,cumulative_oil,cumulative_water,wor,dwor_dcumoil
0,BG-1494P,2020-10-05,0.000000,0.000000,NaN,NaN
1,BG-1494P,2020-10-06,0.000000,0.000000,NaN,NaN
2,BG-1494P,2020-10-07,335.559866,-0.024536,-0.000073,1.900044e-07
3,BG-1494P,2020-10-08,755.423290,0.005029,0.000007,9.003047e-08
4,BG-1494P,2020-10-09,1250.179862,0.064010,0.000051,-1.604745e-07
...,...,...,...,...,...,...
13756,BG-1502P,2021-11-21,244260.140810,268655.264808,1.099874,6.207604e-06
13757,BG-1502P,2021-11-22,244579.693631,269491.894799,1.101857,6.250027e-06
13758,BG-1502P,2021-11-23,244897.650338,270328.907578,1.103844,6.221732e-06
13759,BG-1502P,2021-11-24,245211.875387,271155.158243,1.105799,6.148902e-06


## First, get each table and produce a context for it
An LLM will use that context to select a chart and potentially some previous transformations

In [30]:

# deterministic 
class TableResponseProcessor:
    """
    Builds compact, chart-oriented table summaries for a Plotly presenter LLM.

    The LLM receives metadata and limited column examples, but never the full data.
    """

    def __init__(
        self,
        max_examples: int = 3,
        low_cardinality_threshold: int = 10,
        medium_cardinality_threshold: int = 50,
        categorical_numeric_threshold: int = 12,
    ):
        self.max_examples = max_examples
        self.low_cardinality_threshold = low_cardinality_threshold
        self.medium_cardinality_threshold = medium_cardinality_threshold
        self.categorical_numeric_threshold = categorical_numeric_threshold

    def process(
        self,
        items: Iterable[tuple["TableCard", pd.DataFrame]],
    ) -> str:
        """
        Process multiple (TableCard, DataFrame) pairs into one text block
        for the presenter prompt.
        """

        blocks: list[str] = []

        for table_card, df in items:
            blocks.append(self.process_table(df=df, table_card=table_card))

        #return "\n\n" + ("=" * 80) + "\n\n".join(blocks)
        return ("\n\n" + "=" * 80 + "\n\n").join(blocks)

    def process_table(
        self,
        df: pd.DataFrame,
        table_card: "TableCard",
    ) -> str:
        """
        Convert one DataFrame + TableCard into compact, LLM-friendly text.
        """

        table_name = getattr(table_card, "name", "unknown_table")
        table_description = getattr(table_card, "description", None)

        lines: list[str] = []

        lines.append(f"Table: {table_name}")

        if table_description:
            lines.append(f"Description: {table_description}")

        lines.append(f"Rows: {len(df)}")
        lines.append(f"Columns: {len(df.columns)}")

        lines.append("")
        lines.append("Column summaries:")

        for col in df.columns:
            summary = self.infer_column_summary(df[col])

            description = self.get_column_description(table_card, col)
            if description:
                summary["description"] = description

            lines.append(f"- Column: {col}")

            preferred_order = [
                "description",
                "dtype",
                "role",
                "unique_count",
                "unique_ratio",
                "cardinality",
                "null_count",
                "null_ratio",
                "min",
                "max",
                "mean",
                "median",
                "min_date",
                "max_date",
                "common_interval",
                "is_monotonic_increasing",
                "has_negative_values",
                "has_zero_values",
                "example_values",
            ]

            for key in preferred_order:
                if key in summary:
                    lines.append(f"  {key}: {summary[key]}")

        return "\n".join(lines)

    def infer_column_role(self, s: pd.Series) -> str:
        """
        Infer a chart-oriented semantic role.
        """

        if pd.api.types.is_datetime64_any_dtype(s):
            return "temporal"

        if pd.api.types.is_bool_dtype(s):
            return "categorical"

        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)

            if nunique <= self.categorical_numeric_threshold:
                return "categorical_numeric"

            return "quantitative"

        if pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s):
            parsed = pd.to_datetime(s.dropna(), errors="coerce")
            valid_ratio = parsed.notna().mean() if len(parsed) else 0.0

            if valid_ratio >= 0.9:
                return "temporal"

            return "categorical"

        return "unknown"

    def infer_column_summary(self, s: pd.Series) -> dict[str, Any]:
        """
        Produce compact metadata for one DataFrame column.
        """

        non_null = s.dropna()

        row_count = len(s)
        non_null_count = len(non_null)
        null_count = row_count - non_null_count
        null_ratio = null_count / row_count if row_count else 0.0

        unique_count = non_null.nunique(dropna=True)
        unique_ratio = unique_count / row_count if row_count else 0.0

        role = self.infer_column_role(s)

        summary: dict[str, Any] = {
            "dtype": str(s.dtype),
            "role": role,
            "non_null_count": int(non_null_count),
            "null_count": int(null_count),
            "null_ratio": round(null_ratio, 4),
            "unique_count": int(unique_count),
            "unique_ratio": round(unique_ratio, 4),
        }

        if role in {"categorical", "categorical_numeric"}:
            summary["cardinality"] = self.classify_cardinality(unique_count)

            examples = (
                non_null
                .drop_duplicates()
                .astype(str)
                .head(self.max_examples)
                .tolist()
            )

            if examples:
                summary["example_values"] = examples

        elif role == "quantitative":
            numeric = pd.to_numeric(non_null, errors="coerce").dropna()

            if not numeric.empty:
                summary.update(
                    {
                        "min": round(float(numeric.min()), 4),
                        "max": round(float(numeric.max()), 4),
                        "mean": round(float(numeric.mean()), 4),
                        "median": round(float(numeric.median()), 4),
                        "has_negative_values": bool((numeric < 0).any()),
                        "has_zero_values": bool((numeric == 0).any()),
                    }
                )

        elif role == "temporal":
            dt = pd.to_datetime(non_null, errors="coerce").dropna()

            if not dt.empty:
                summary.update(
                    {
                        "min_date": str(dt.min()),
                        "max_date": str(dt.max()),
                        "is_monotonic_increasing": bool(dt.is_monotonic_increasing),
                    }
                )

                if len(dt) > 1:
                    diffs = dt.sort_values().diff().dropna()
                    if not diffs.empty:
                        mode_diff = diffs.mode()
                        if not mode_diff.empty:
                            summary["common_interval"] = str(mode_diff.iloc[0])

        return summary

    def classify_cardinality(self, unique_count: int) -> str:
        """
        Convert unique count into low/medium/high cardinality label.
        """

        if unique_count <= self.low_cardinality_threshold:
            return "low"

        if unique_count <= self.medium_cardinality_threshold:
            return "medium"

        return "high"

    def get_column_description(
        self,
        table_card: "TableCard",
        column_name: str,
    ) -> str | None:
        """
        Extract column description from a TableCard, if present.

        Supports columns represented as either objects or dictionaries.
        """

        columns = getattr(table_card, "columns", None)

        if not columns:
            return None

        for col in columns:
            if isinstance(col, dict):
                name = col.get("name")
                description = col.get("description")
            else:
                name = getattr(col, "name", None)
                description = getattr(col, "description", None)

            if name == column_name:
                return description

        return None
   

In [31]:
r = response['structured_response']
table_name =r.tables[0].table_name
df = smart_data.get_table_as_df(table_name)

derived = smart_data.catalog_snapshot(table_name).derived_tables

table_card = [ t for t in derived if t.name == table_name][0]
user_query =response['structured_response'].user_query
#user_query = "Plot total water injection volume per year and subzone"
#user_query = "Plot maximum quarterly water injection volume per year and subzone"
#user_query = "Plot total water injection volume by year and subzone"
p = TableResponseProcessor() 
table_context = p.process([(table_card, df)])
pprint.pprint(table_context[0:100])


('Table: water_oil_ratio_and_derivative\n'
 'Description: This table contains the Water-Oil Ratio (WOR) and')


C:\Users\xteijeiro\AppData\Local\Temp\ipykernel_25840\3409260552.py:120: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [43]:
from langchain_core.messages import SystemMessage, HumanMessage

  
CHART_AGENT_PROMPTV1 = """
You are a chart planning agent.

You receive:
- user query
- table summaries
- column names, roles, cardinality, and descriptions

Return a JSON plan with:
- optional preprocess step
- exactly one plot step

PREPROCESS TOOL

preprocess_for_chart:
Use only when a needed chart column can be derived safely.

Args:
{
  "create_combined_category": null | {
    "col1": "<categorical_col>",
    "col2": "<categorical_col>",
    "new_col": "<new_col>",
    "sep": " / "
  },
  "create_date_bucket": null | {
    "date_col": "<date_col>",
    "bucket": "D|W|M|Q|Y",
    "new_col": "<new_col>"
  }
}

PLOT TOOLS

plot_bar_chart:
Use for quantitative values or counts compared across categorical or bucketed temporal dimensions.
Args:
{
  "x": "<category_or_bucket_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>"],
  "color_by": null | "<secondary_category_col>",
  "orientation": "v|h",
  "barmode": "group|stack|relative",
  "title": "<title>"
}

plot_line_chart:
Use for trends, time series, ordered progression, or cumulative values.
Args:
{
  "x": "<time_or_ordered_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>"],
  "color_by": null | "<category_col>",
  "date_bucket": null | "D|W|M|Q|Y",
  "cumulative": true|false,
  "title": "<title>"
}

plot_pie_chart:
Use only for part-to-whole/share/composition questions.
Args:
{
  "labels": "<category_col>",
  "values": "<numeric_col>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<label_col>"],
  "hole": 0.0,
  "title": "<title>"
}

plot_scatter_chart:
Use for numeric-vs-numeric relationships, correlations, crossplots, clusters, or row-level comparisons.
Args:
{
  "x": "<numeric_col>",
  "y": "<numeric_col_or_list>",
  "color_by": null | "<category_col>",
  "size_by": null | "<numeric_col>",
  "text_by": null | "<label_col>",
  "title": "<title>"
}

RULES
- Return only valid JSON.
- Do not invent tools.
- Do not invent arguments.
- Use only columns that exist or are created by preprocess_for_chart.
- Prefer no preprocess when existing columns are sufficient.
- Use sum by default for additive quantities unless otherwise specified.
- If uncertain, return {"reason": "...", "preprocess": null, "plot": null}.
- When multiple temporal dimensions together define the displayed x-axis grouping
(e.g. year + quarter, year + month),
create a combined temporal category for x.

OUTPUT SHAPE
{
  "reason": "<brief reason>",
  "preprocess": null | {
    "tool": "preprocess_for_chart",
    "args": {}
  },
  "plot": null | {
    "tool": "<plot_tool>",
    "args": {}
  }
}
"""

CHART_AGENT_PROMPTV2 = """
You are a chart planning agent.

You receive:
- user query
- table summaries
- column names, roles, cardinality, and descriptions

Return a JSON plan with:
- optional preprocess step
- exactly one plot step

 
preprocess_for_chart:

Supported operation:

1. create_combined_category
Creates one new text/category column by concatenating two existing columns.
Use it only when the plot needs a display/grouping column that is not already present but can be safely created from existing columns.
Use when:
- Two columns together define the chart category or x-axis label.
- A single readable display label is needed for plotting.
- The user asks for a breakdown involving two dimensions that should appear as one chart category.



Use only when a needed chart column can be derived safely.

Args:
{
  "create_combined_category": null | {
    "col1": "<categorical_col>",
    "col2": "<categorical_col>",
    "new_col": "<new_col>",
    "sep": " / "
  },

}

PLOT TOOLS

plot_bar_chart:
Use for comparing one or more quantitative values across categorical or bucketed temporal groups.

Best for:
- "Y by A"
- "Y per A"
- "Y by A and B"
- totals, averages, counts, rankings, grouped comparisons

Mapping rules:
- For "Y by A": use x = A, y = Y, group_by = [A].
- For "Y by A and B": use x = A, color_by = B, y = Y, group_by = [A, B].
- For "Y by A, B, and C": use x = A, color_by = B or C, and group_by = [A, B, C].
- If two columns together define the x-axis label, create the combined column first with preprocess_for_chart and use it as x.
- group_by must include every column needed to preserve the requested breakdown.
- Use aggregate = "sum" by default for additive quantities unless the query specifies another aggregation.

Args:
{
  "x": "<category_or_bucket_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>"],
  "color_by": null | "<secondary_category_col>",
  "orientation": "v|h",
  "barmode": "group|stack|relative",
  "title": "<title>"
}

plot_line_chart:
Use for trends, time series, ordered progression, or cumulative values.
Args:
{
  "x": "<time_or_ordered_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>"],
  "color_by": null | "<category_col>",
  "date_bucket": null | "D|W|M|Q|Y",
  "cumulative": true|false,
  "title": "<title>"
}

plot_pie_chart:
Use only for part-to-whole/share/composition questions.
Args:
{
  "labels": "<category_col>",
  "values": "<numeric_col>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<label_col>"],
  "hole": 0.0,
  "title": "<title>"
}

plot_scatter_chart:
Use for numeric-vs-numeric relationships, correlations, crossplots, clusters, or row-level comparisons.
Args:
{
  "x": "<numeric_col>",
  "y": "<numeric_col_or_list>",
  "color_by": null | "<category_col>",
  "size_by": null | "<numeric_col>",
  "text_by": null | "<label_col>",
  "title": "<title>"
}

RULES
- Return only valid JSON.
- Do not invent tools.
- Do not invent arguments.
- Use only columns that exist or are created by preprocess_for_chart.
- Prefer no preprocess when existing columns are sufficient.
- Use sum by default for additive quantities unless otherwise specified.
- If uncertain, return {"reason": "...", "preprocess": null, "plot": null}.
- When multiple temporal dimensions together define the displayed x-axis grouping
(e.g. year + quarter, year + month),
create a combined temporal category for x.

OUTPUT SHAPE
{
  "reason": "<brief reason>",
  "preprocess": null | {
    "tool": "preprocess_for_chart",
    "args": {}
  },
  "plot": null | {
    "tool": "<plot_tool>",
    "args": {}
  }
}
"""


CHART_AGENT_PROMPT = CHART_AGENT_PROMPTV2

In [52]:

#response['structured_response']
#to call the processor we need to pass it a list of (TableCard, DataFrame) pairs. The TableCard can be obtained from the agent's response, and the DataFrame can be retrieved from the SmartData catalog using the table name provided in the response.

p = TableResponseProcessor() 
table_context = p.process([(table, df)])


# ============================================================
# Replace select_chart_plan with this
# ============================================================
import re
def format_label(name: str) -> str:
    """
    Convert column-like names to display labels.

    Examples:
    - year_quarter -> Year quarter
    - percentage_contribution -> Percentage contribution
    - TOTAL_WATER_INJECTION_VOLUME -> Total water injection volume
    """
    if name is None:
        return ""

    text = str(name).replace("_", " ").strip().lower()
    return text[:1].upper() + text[1:]
def _as_list(value):
    if value is None:
        return []
    return [value] if isinstance(value, str) else list(value)

def _strip_markdown_json(text: str) -> str:
    """
    Remove markdown code fences from LLM JSON responses.

    Examples:
    ```json
    {...}
    ```

    ->
    {...}
    """

    text = text.strip()

    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return text.strip()

def select_chart_plan(
    llm,
    user_query: str,
    table_context: str,
) -> dict:
    from langchain_core.messages import SystemMessage, HumanMessage

    messages = [
        SystemMessage(content=CHART_AGENT_PROMPT),
        HumanMessage(content=f"""
USER QUERY
{user_query}

TABLES
{table_context}
"""),
    ]

    response = llm.invoke(messages)
    text = _strip_markdown_json(response.content)
    return json.loads(text)

def run_chart_plan(
    plan: dict,
    df: pd.DataFrame,
) -> dict | None:
    """
    Executes a chart plan shaped like:

    {
      "reason": "...",
      "preprocess": null | {
        "tool": "preprocess_for_chart",
        "args": {}
      },
      "plot": null | {
        "tool": "plot_bar_chart",
        "args": {}
      }
    }
    """

    work = df.copy()

    preprocess = plan.get("preprocess")
    plot = plan.get("plot")

    if preprocess:
        tool = preprocess.get("tool")
        args = preprocess.get("args") or {}

        if tool != "preprocess_for_chart":
            raise ValueError(f"Unknown preprocess tool: {tool}")

        args = _filter_args("preprocess_for_chart", args)
        work = preprocess_for_chart(work, **args)

    if not plot:
        return None

    tool = plot.get("tool")
    args = plot.get("args") or {}

    plotting_tools = {
        "plot_bar_chart": plot_bar_chart,
        "plot_line_chart": plot_line_chart,
        "plot_pie_chart": plot_pie_chart,
        "plot_scatter_chart": plot_scatter_chart,
    }

    if tool not in plotting_tools:
        raise ValueError(f"Unknown plot tool: {tool}")

    args = _filter_args(tool, args)

    return plotting_tools[tool](df=work, **args)

def _validate_columns(
    df: pd.DataFrame,
    columns: list[str],
    label: str = "column",
):
    """
    Validate that all requested columns exist in the dataframe.
    """

    missing = [c for c in columns if c not in df.columns]

    if missing:
        raise ValueError(f"Missing {label}(s): {missing}")

def _filter_args(
    tool: str,
    args: dict[str, Any],
) -> dict[str, Any]:
    """
    Remove unsupported arguments generated by the LLM.
    """

    allowed_args = {
        "preprocess_for_chart": {
            "create_combined_category",
            "create_date_bucket",
        },

        "plot_bar_chart": {
            "x",
            "y",
            "aggregate",
            "group_by",
            "color_by",
            "orientation",
            "barmode",
            "title",
            "template",
        },

        "plot_line_chart": {
            "x",
            "y",
            "aggregate",
            "group_by",
            "color_by",
            "date_bucket",
            "cumulative",
            "title",
            "template",
        },

        "plot_pie_chart": {
            "labels",
            "values",
            "aggregate",
            "group_by",
            "title",
            "hole",
            "template",
        },

        "plot_scatter_chart": {
            "x",
            "y",
            "color_by",
            "size_by",
            "text_by",
            "title",
            "template",
        },
    }

    if tool not in allowed_args:
        raise ValueError(f"Unknown tool: {tool}")

    return {
        k: v
        for k, v in args.items()
        if k in allowed_args[tool]
    }

ALLOWED_AGGS = {"sum", "mean", "median", "min", "max", "count", "nunique"}


def _aggregate(
    df: pd.DataFrame,
    group_by: list[str],
    value_cols: list[str],
    aggregate: str,
) -> pd.DataFrame:
    if aggregate not in ALLOWED_AGGS:
        raise ValueError(f"Unsupported aggregate: {aggregate}")

    _validate_columns(df, group_by, "group_by column")
    _validate_columns(df, value_cols, "value column")

    return (
        df.groupby(group_by, dropna=False, as_index=False)[value_cols]
        .agg(aggregate)
    )


def _bucket_date(
    df: pd.DataFrame,
    date_col: str,
    bucket: str,
) -> tuple[pd.DataFrame, str]:
    """
    Create a date bucket column.

    bucket:
    - "D": day
    - "W": week
    - "M": month
    - "Q": quarter
    - "Y": year
    """

    out = df.copy()
    bucket_col = f"{date_col}_{bucket}"

    _validate_columns(out, [date_col])

    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

    if bucket == "D":
        out[bucket_col] = out[date_col].dt.to_period("D").dt.to_timestamp()
    elif bucket == "W":
        out[bucket_col] = out[date_col].dt.to_period("W").dt.start_time
    elif bucket == "M":
        out[bucket_col] = out[date_col].dt.to_period("M").dt.to_timestamp()
    elif bucket == "Q":
        out[bucket_col] = out[date_col].dt.to_period("Q").dt.to_timestamp()
    elif bucket == "Y":
        out[bucket_col] = out[date_col].dt.to_period("Y").dt.to_timestamp()
    else:
        raise ValueError("date_bucket must be one of: D, W, M, Q, Y")

    return out, bucket_col

def preprocess_for_chart(
    df: pd.DataFrame,
    *,
    create_combined_category: dict | None = None,
    create_date_bucket: dict | None = None,
) -> pd.DataFrame:
    """
    Prepare a dataframe for charting by creating optional derived columns.
    """

    out = df.copy()

    if create_combined_category:
        col1 = create_combined_category["col1"]
        col2 = create_combined_category["col2"]
        new_col = create_combined_category.get("new_col") or f"{col1}_{col2}"
        sep = create_combined_category.get("sep", " / ")

        _validate_columns(out, [col1, col2])

        out[new_col] = (
            out[col1].fillna("").astype(str)
            + sep
            + out[col2].fillna("").astype(str)
        )

    if create_date_bucket:
        date_col = create_date_bucket["date_col"]
        bucket = create_date_bucket["bucket"]
        new_col = create_date_bucket.get("new_col") or f"{date_col}_{bucket}"

        _validate_columns(out, [date_col])

        dt = pd.to_datetime(out[date_col], errors="coerce")

        if bucket == "D":
            out[new_col] = dt.dt.to_period("D").dt.to_timestamp()
        elif bucket == "W":
            out[new_col] = dt.dt.to_period("W").dt.start_time
        elif bucket == "M":
            out[new_col] = dt.dt.to_period("M").dt.to_timestamp()
        elif bucket == "Q":
            out[new_col] = dt.dt.to_period("Q").dt.to_timestamp()
        elif bucket == "Y":
            out[new_col] = dt.dt.to_period("Y").dt.to_timestamp()
        else:
            raise ValueError("bucket must be one of: D, W, M, Q, Y")

    return out


# ============================================================
# Chart tools
# Requires: pandas as pd, _as_list, _validate_columns, _aggregate, _bucket_date
# ============================================================

def xxplot_bar_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    aggregate: str | None = None,
    group_by: list[str] | str | None = None,
    color_by: str | None = None,
    orientation: str = "v",
    barmode: str = "group",
    title: str | None = None,
    template: str = "plotly_white"
) -> dict:
    y_cols = _as_list(y)

    required = [x, *y_cols]
    if color_by:
        required.append(color_by)

    _validate_columns(df, required)

    work = df.copy()

    if aggregate is not None:
        group_cols = _as_list(group_by) or [x]

        if x not in group_cols:
            group_cols.insert(0, x)

        if color_by and color_by not in group_cols:
            group_cols.append(color_by)

        work = _aggregate(work, group_cols, y_cols, aggregate)

    data = []
    groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

    for group_value, g in groups:
        for y_col in y_cols:
            name = y_col if group_value is None else str(group_value)

            if group_value is not None and len(y_cols) > 1:
                name = f"{group_value} - {y_col}"

            trace = {
                "type": "bar",
                "name": name,
                "orientation": orientation,
            }

            if orientation == "h":
                trace["x"] = g[y_col].tolist()
                trace["y"] = g[x].astype(str).tolist()
            else:
                trace["x"] = g[x].astype(str).tolist()
                trace["y"] = g[y_col].tolist()

            data.append(trace)

    return {
        "data": data,
        "layout": {
    "title": {"text": title or f"{format_label(', '.join(y_cols))} by {format_label(x)}"},
    "xaxis": {"title": {"text": format_label(", ".join(y_cols)) if orientation == "h" else format_label(x)}},
    "yaxis": {"title": {"text": format_label(x) if orientation == "h" else format_label(", ".join(y_cols))}},
    "barmode": barmode,
    "template": template,
},
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


def plot_bar_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    aggregate: str | None = None,
    group_by: list[str] | str | None = None,
    color_by: str | None = None,
    orientation: str = "v",
    barmode: str = "group",
    title: str | None = None,
    template: str = "plotly_white",
) -> dict:
    y_cols = _as_list(y)

    required = [x, *y_cols]
    if color_by:
        required.append(color_by)

    _validate_columns(df, required)

    work = df.copy()

    if aggregate is not None:
        group_cols = _as_list(group_by) or [x]

        if x not in group_cols:
            group_cols.insert(0, x)

        if color_by and color_by not in group_cols:
            group_cols.append(color_by)

        work = _aggregate(work, group_cols, y_cols, aggregate)

    data = []
    groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

    for group_value, g in groups:
        for y_col in y_cols:
            if group_value is None:
                name = format_label(y_col)
            else:
                name = format_label(str(group_value))

            if group_value is not None and len(y_cols) > 1:
                name = f"{format_label(str(group_value))} - {format_label(y_col)}"

            trace = {
                "type": "bar",
                "name": name,
                "orientation": orientation,
            }

            if orientation == "h":
                trace["x"] = g[y_col].tolist()
                trace["y"] = g[x].astype(str).tolist()
            else:
                trace["x"] = g[x].astype(str).tolist()
                trace["y"] = g[y_col].tolist()

            data.append(trace)

    y_label = format_label(", ".join(y_cols))
    x_label = format_label(x)

    return {
        "data": data,
        "layout": {
            "title": {
                "text": title or f"{y_label} by {x_label}"
            },
            "xaxis": {
                "title": {
                    "text": y_label if orientation == "h" else x_label
                }
            },
            "yaxis": {
                "title": {
                    "text": x_label if orientation == "h" else y_label
                }
            },
            "barmode": barmode,
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }

def plot_line_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    aggregate: str | None = None,
    group_by: list[str] | str | None = None,
    color_by: str | None = None,
    date_bucket: str | None = None,
    cumulative: bool = False,
    title: str | None = None,
    template: str = "plotly_white",
) -> dict:
    y_cols = _as_list(y)

    required = [x, *y_cols]
    if color_by:
        required.append(color_by)

    _validate_columns(df, required)

    work = df.copy()
    x_plot = x

    if date_bucket is not None:
        work, x_plot = _bucket_date(work, x, date_bucket)

    if aggregate is not None:
        group_cols = _as_list(group_by) or [x_plot]

        if x_plot not in group_cols:
            group_cols.insert(0, x_plot)

        if color_by and color_by not in group_cols:
            group_cols.append(color_by)

        work = _aggregate(work, group_cols, y_cols, aggregate)

    sort_cols = [color_by, x_plot] if color_by else [x_plot]
    work = work.sort_values(sort_cols)

    if cumulative:
        if color_by:
            for col in y_cols:
                work[col] = work.groupby(color_by, dropna=False)[col].cumsum()
        else:
            for col in y_cols:
                work[col] = work[col].cumsum()

    data = []
    groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

    
    total_points = len(work) * len(y_cols)
    disable_all_markers = total_points > 2000


    for group_value, g in groups:
        for y_col in y_cols:
            name = y_col if group_value is None else str(group_value)

            if group_value is not None and len(y_cols) > 1:
                name = f"{group_value} - {y_col}"



            n_points = len(g)

            use_markers = (
                not disable_all_markers
                and n_points <= 100
            )

            mode = "lines"##"lines+markers" if use_markers else "lines"


            data.append({
                "type": "scatter",
                "mode": mode,#"lines+markers",
                "x": g[x_plot].tolist(),
                "y": g[y_col].tolist(),
                "name": name,
            })

    return {
        "data": data,
        "layout": {
            "title": {"text": title or f"{', '.join(y_cols)} over {x}"},
            "xaxis": {"title": {"text": x}},
            "yaxis": {"title": {"text": ", ".join(y_cols)}},
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


def plot_pie_chart(
    df: pd.DataFrame,
    labels: str,
    values: str,
    *,
    aggregate: str | None = None,
    group_by: list[str] | str | None = None,
    title: str | None = None,
    hole: float = 0.0,
    template: str = "plotly_white",
) -> dict:
    _validate_columns(df, [labels, values])

    work = df.copy()

    if aggregate is not None:
        group_cols = _as_list(group_by) or [labels]

        if labels not in group_cols:
            group_cols.insert(0, labels)

        work = _aggregate(work, group_cols, [values], aggregate)

    return {
        "data": [
            {
                "type": "pie",
                "labels": work[labels].astype(str).tolist(),
                "values": work[values].tolist(),
                "hole": hole,
            }
        ],
        "layout": {
            "title": {"text": title or f"{values} share by {labels}"},
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


def plot_scatter_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    color_by: str | None = None,
    size_by: str | None = None,
    text_by: str | None = None,
    title: str | None = None,
    template: str = "plotly_white",
) -> dict:
    y_cols = _as_list(y)

    required = [x, *y_cols]
    if color_by:
        required.append(color_by)
    if size_by:
        required.append(size_by)
    if text_by:
        required.append(text_by)

    _validate_columns(df, required)

    work = df.copy()
    data = []
    groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]


    total_points = len(work) * len(y_cols)
    disable_all_markers = total_points > 2000


    for group_value, g in groups:
        for y_col in y_cols:
            name = y_col if group_value is None else str(group_value)

            if group_value is not None and len(y_cols) > 1:
                name = f"{group_value} - {y_col}"




            n_points = len(g)

            use_markers = (
                not disable_all_markers
                and n_points <= 100
            )

            mode = "markers" if use_markers else "lines"


            trace = {
                "type": "scattergl",
                "mode": mode,
                "x": g[x].tolist(),
                "y": g[y_col].tolist(),
                "name": name,
            }

            if size_by:
                size_values = pd.to_numeric(g[size_by], errors="coerce").fillna(0)
                max_size = max(float(size_values.max()), 1.0)

                trace["marker"] = {
                    "size": size_values.tolist(),
                    "sizemode": "area",
                    "sizeref": max_size / 40,
                    "sizemin": 4,
                }

            if text_by:
                trace["text"] = g[text_by].astype(str).tolist()
                trace["hovertemplate"] = (
                    f"{x}: %{{x}}<br>"
                    f"{y_col}: %{{y}}<br>"
                    f"{text_by}: %{{text}}"
                    "<extra></extra>"
                )

            data.append(trace)

    return {
        "data": data,
        "layout": {
            "title": {"text": title or f"{', '.join(y_cols)} vs {x}"},
            "xaxis": {"title": {"text": x}},
            "yaxis": {"title": {"text": ", ".join(y_cols)}},
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }



C:\Users\xteijeiro\AppData\Local\Temp\ipykernel_25840\3409260552.py:120: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [53]:

llm = azure_llm_if()

plan = select_chart_plan(llm, user_query, table_context)
print(json.dumps(plan, indent=2))



{
  "reason": "The user wants to plot the Water-Oil Ratio (WOR) and its derivative with respect to cumulative oil production on a log-log scale, with each producer represented as a separate trace.",
  "preprocess": null,
  "plot": {
    "tool": "plot_scatter_chart",
    "args": {
      "x": "cumulative_oil",
      "y": [
        "wor",
        "dwor_dcumoil"
      ],
      "color_by": "NAME",
      "title": "Water-Oil Ratio (WOR) and its Derivative vs Cumulative Oil Production (Log-Log Scale)"
    }
  }
}


In [54]:
fig = run_chart_plan(plan, df)

import plotly.io as pio
pio.show(fig)

### We need to be abe to change the data, and keep the smart data and the agent

Lets first ask a question about the current dataset

In [ ]:

simple_question = "How many injectors are there?"
messages = {"messages": [{"role": "user", "content": simple_question}]}
response = agent.invoke(messages, config={"recursion_limit": 20})
response['structured_response']
r = response['structured_response']
table_name =r.tables[0].table_name
df = smart_data.get_table_as_df(table_name)
df


: 

Now lets chage the data. Update the the smart data and ask again 

In [ ]:
path =  "../datasets/IX5I_4P/"
inj, pinj, locs = fetch_data(path)

smart_data.set_data({'injectors': inj, 'producers': pinj, 'locations': locs})


: 

In [ ]:
simple_question = "How many injectors are there?"
messages = {"messages": [{"role": "user", "content": simple_question}]}
response = agent.invoke(messages, config={"recursion_limit": 20})
response['structured_response']
r = response['structured_response']
table_name =r.tables[0].table_name
df = smart_data.get_table_as_df(table_name)
display( df.head() )


: 

In [ ]:
path =  "../datasets/Demo1/"
inj, pinj, locs = fetch_data(path)
smart_data.set_data({'injectors': inj, 'producers': pinj, 'locations': locs})
simple_question = "How many injectors are there?"
messages = {"messages": [{"role": "user", "content": simple_question}]}
response = agent.invoke(messages, config={"recursion_limit": 20})
response['structured_response']
r = response['structured_response']
table_name =r.tables[0].table_name
df = smart_data.get_table_as_df(table_name)
display( df.head() )


: 

In [ ]:
# Experiment, what if the agent reponse can accommodate, text, table and question (terminal)

: 

: 

: 

: 

: 

: 

: 

In [ ]:

class TableResponseProcessor:
    """
    Builds compact, chart-oriented table summaries for a Plotly presenter LLM.
    The LLM receives metadata and limited column examples, but never the full data.
    """

    def __init__(
        self,
        max_examples: int = 3,
        low_cardinality_threshold: int = 10,
        medium_cardinality_threshold: int = 50,
        categorical_numeric_threshold: int = 12,
    ):
        self.max_examples = max_examples
        self.low_cardinality_threshold = low_cardinality_threshold
        self.medium_cardinality_threshold = medium_cardinality_threshold
        self.categorical_numeric_threshold = categorical_numeric_threshold

    def process(
        self,
        items: Iterable[tuple["TableCard", pd.DataFrame]],
    ) -> str:
        """
        Process multiple (TableCard, DataFrame) pairs into one text block
        for the presenter prompt.
        """

        blocks: list[str] = []

        for table_card, df in items:
            blocks.append(self.process_table(df=df, table_card=table_card))

        #return "\n\n" + ("=" * 80) + "\n\n".join(blocks)
        return ("\n\n" + "=" * 80 + "\n\n").join(blocks)

    def process_table(
        self,
        df: pd.DataFrame,
        table_card: "TableCard",
    ) -> str:
        """
        Convert one DataFrame + TableCard into compact, LLM-friendly text.
        """

        table_name = getattr(table_card, "name", "unknown_table")
        table_description = getattr(table_card, "description", None)

        lines: list[str] = []

        lines.append(f"Table: {table_name}")

        if table_description:
            lines.append(f"Description: {table_description}")

        lines.append(f"Rows: {len(df)}")
        lines.append(f"Columns: {len(df.columns)}")

        lines.append("")
        lines.append("Column summaries:")

        for col in df.columns:
            summary = self.infer_column_summary(df[col])

            description = self.get_column_description(table_card, col)
            if description:
                summary["description"] = description

            lines.append(f"- Column: {col}")

            preferred_order = [
                "description",
                "dtype",
                "role",
                "unique_count",
                "unique_ratio",
                "cardinality",
                "null_count",
                "null_ratio",
                "min",
                "max",
                "mean",
                "median",
                "min_date",
                "max_date",
                "common_interval",
                "is_monotonic_increasing",
                "has_negative_values",
                "has_zero_values",
                "example_values",
            ]

            for key in preferred_order:
                if key in summary:
                    lines.append(f"  {key}: {summary[key]}")

        return "\n".join(lines)

    def infer_column_role(self, s: pd.Series) -> str:
        """
        Infer a chart-oriented semantic role.
        """

        if pd.api.types.is_datetime64_any_dtype(s):
            return "temporal"

        if pd.api.types.is_bool_dtype(s):
            return "categorical"

        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)

            if nunique <= self.categorical_numeric_threshold:
                return "categorical_numeric"

            return "quantitative"

        if pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s):
            parsed = pd.to_datetime(s.dropna(), errors="coerce")
            valid_ratio = parsed.notna().mean() if len(parsed) else 0.0

            if valid_ratio >= 0.9:
                return "temporal"

            return "categorical"

        return "unknown"

    def infer_column_summary(self, s: pd.Series) -> dict[str, Any]:
        """
        Produce compact metadata for one DataFrame column.
        """

        non_null = s.dropna()

        row_count = len(s)
        non_null_count = len(non_null)
        null_count = row_count - non_null_count
        null_ratio = null_count / row_count if row_count else 0.0

        unique_count = non_null.nunique(dropna=True)
        unique_ratio = unique_count / row_count if row_count else 0.0

        role = self.infer_column_role(s)

        summary: dict[str, Any] = {
            "dtype": str(s.dtype),
            "role": role,
            "non_null_count": int(non_null_count),
            "null_count": int(null_count),
            "null_ratio": round(null_ratio, 4),
            "unique_count": int(unique_count),
            "unique_ratio": round(unique_ratio, 4),
        }

        if role in {"categorical", "categorical_numeric"}:
            summary["cardinality"] = self.classify_cardinality(unique_count)

            examples = (
                non_null
                .drop_duplicates()
                .astype(str)
                .head(self.max_examples)
                .tolist()
            )

            if examples:
                summary["example_values"] = examples

        elif role == "quantitative":
            numeric = pd.to_numeric(non_null, errors="coerce").dropna()

            if not numeric.empty:
                summary.update(
                    {
                        "min": round(float(numeric.min()), 4),
                        "max": round(float(numeric.max()), 4),
                        "mean": round(float(numeric.mean()), 4),
                        "median": round(float(numeric.median()), 4),
                        "has_negative_values": bool((numeric < 0).any()),
                        "has_zero_values": bool((numeric == 0).any()),
                    }
                )

        elif role == "temporal":
            dt = pd.to_datetime(non_null, errors="coerce").dropna()

            if not dt.empty:
                summary.update(
                    {
                        "min_date": str(dt.min()),
                        "max_date": str(dt.max()),
                        "is_monotonic_increasing": bool(dt.is_monotonic_increasing),
                    }
                )

                if len(dt) > 1:
                    diffs = dt.sort_values().diff().dropna()
                    if not diffs.empty:
                        mode_diff = diffs.mode()
                        if not mode_diff.empty:
                            summary["common_interval"] = str(mode_diff.iloc[0])

        return summary

    def classify_cardinality(self, unique_count: int) -> str:
        """
        Convert unique count into low/medium/high cardinality label.
        """

        if unique_count <= self.low_cardinality_threshold:
            return "low"

        if unique_count <= self.medium_cardinality_threshold:
            return "medium"

        return "high"

    def get_column_description(
        self,
        table_card: "TableCard",
        column_name: str,
    ) -> str | None:
        """
        Extract column description from a TableCard, if present.

        Supports columns represented as either objects or dictionaries.
        """

        columns = getattr(table_card, "columns", None)

        if not columns:
            return None

        for col in columns:
            if isinstance(col, dict):
                name = col.get("name")
                description = col.get("description")
            else:
                name = getattr(col, "name", None)
                description = getattr(col, "description", None)

            if name == column_name:
                return description

        return None
    





: 

In [ ]:
from __future__ import annotations

from typing import Any
import json
import pandas as pd
import numpy as np


# ============================================================
# Shared helpers
# ============================================================

ALLOWED_AGGS = {"sum", "mean", "median", "min", "max", "count", "nunique"}

def format_label(name: str) -> str:
    """
    Convert column-like names to display labels.

    Examples:
    - year_quarter -> Year quarter
    - percentage_contribution -> Percentage contribution
    - TOTAL_WATER_INJECTION_VOLUME -> Total water injection volume
    """
    if name is None:
        return ""

    text = str(name).replace("_", " ").strip().lower()
    return text[:1].upper() + text[1:]

def _as_list(value):
    if value is None:
        return []
    return [value] if isinstance(value, str) else list(value)


def _validate_columns(df: pd.DataFrame, columns: list[str], label: str = "column"):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"Missing {label}(s): {missing}")


def _aggregate(
    df: pd.DataFrame,
    group_by: list[str],
    value_cols: list[str],
    aggregate: str,
) -> pd.DataFrame:
    if aggregate not in ALLOWED_AGGS:
        raise ValueError(f"Unsupported aggregate: {aggregate}")

    _validate_columns(df, group_by, "group_by column")
    _validate_columns(df, value_cols, "value column")

    return (
        df.groupby(group_by, dropna=False, as_index=False)[value_cols]
        .agg(aggregate)
    )


def _bucket_date(
    df: pd.DataFrame,
    date_col: str,
    bucket: str,
) -> tuple[pd.DataFrame, str]:
    """
    bucket: D, W, M, Q, Y
    """
    out = df.copy()
    bucket_col = f"{date_col}_{bucket}"

    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

    if bucket == "D":
        out[bucket_col] = out[date_col].dt.to_period("D").dt.to_timestamp()
    elif bucket == "W":
        out[bucket_col] = out[date_col].dt.to_period("W").dt.start_time
    elif bucket == "M":
        out[bucket_col] = out[date_col].dt.to_period("M").dt.to_timestamp()
    elif bucket == "Q":
        out[bucket_col] = out[date_col].dt.to_period("Q").dt.to_timestamp()
    elif bucket == "Y":
        out[bucket_col] = out[date_col].dt.to_period("Y").dt.to_timestamp()
    else:
        raise ValueError("date_bucket must be one of: D, W, M, Q, Y")

    return out, bucket_col


def create_combined_category_column(
    df: pd.DataFrame,
    col1: str,
    col2: str,
    new_col: str | None = None,
    sep: str = " / ",
) -> pd.DataFrame:
    """
    Create a readable combined categorical column from two categorical columns.
    """
    _validate_columns(df, [col1, col2])

    out = df.copy()
    new_col = new_col or f"{col1}_{col2}"

    out[new_col] = (
        out[col1].astype(str).fillna("")
        + sep
        + out[col2].astype(str).fillna("")
    )

    return out


# ============================================================
# Chart tools
# ============================================================

def plot_bar_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    aggregate: str | None = None,
    group_by: list[str] | None = None,
    color_by: str | None = None,
    orientation: str = "v",
    barmode: str = "group",
    title: str | None = None,
    template: str = "plotly_white",
) -> dict:
    """
    Create a Plotly bar chart.

    Use for quantitative values or counts compared across categorical
    or bucketed temporal dimensions.
    """

    y_cols = _as_list(y)
    required = [x, *y_cols] + ([color_by] if color_by else [])
    _validate_columns(df, required)

    work = df.copy()

    if aggregate is not None:
        group_cols = list(group_by or [x])

        if x not in group_cols:
            group_cols.insert(0, x)

        if color_by and color_by not in group_cols:
            group_cols.append(color_by)

        work = _aggregate(work, group_cols, y_cols, aggregate)

    data = []

    groups = (
        work.groupby(color_by, dropna=False)
        if color_by
        else [(None, work)]
    )

    for group_value, g in groups:
        for y_col in y_cols:
            name = y_col if group_value is None else str(group_value)

            if group_value is not None and len(y_cols) > 1:
                name = f"{group_value} - {y_col}"

            trace = {
                "type": "bar",
                "name": name,
                "orientation": orientation,
            }

            if orientation == "h":
                trace["x"] = g[y_col].tolist()
                trace["y"] = g[x].astype(str).tolist()
            else:
                trace["x"] = g[x].astype(str).tolist()
                trace["y"] = g[y_col].tolist()

            data.append(trace)

    return {
        "data": data,
       "layout": {
    "title": {"text": title or f"{format_label(', '.join(y_cols))} by {format_label(x)}"},
    "xaxis": {"title": {"text": format_label(", ".join(y_cols)) if orientation == "h" else format_label(x)}},
    "yaxis": {"title": {"text": format_label(x) if orientation == "h" else format_label(", ".join(y_cols))}},
    "barmode": barmode,
    "template": template,
},
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


def plot_line_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    aggregate: str | None = None,
    group_by: list[str] | None = None,
    color_by: str | None = None,
    date_bucket: str | None = None,   # D, W, M, Q, Y
    cumulative: bool = False,
    title: str | None = None,
    template: str = "plotly_white",
) -> dict:
    """
    Create a Plotly line chart.

    Use for trends, time series, ordered progression, or cumulative values.
    """

    y_cols = _as_list(y)
    required = [x, *y_cols] + ([color_by] if color_by else [])
    _validate_columns(df, required)

    work = df.copy()
    x_plot = x

    if date_bucket is not None:
        work, x_plot = _bucket_date(work, x, date_bucket)

    if aggregate is not None:
        group_cols = list(group_by or [x_plot])

        if x_plot not in group_cols:
            group_cols.insert(0, x_plot)

        if color_by and color_by not in group_cols:
            group_cols.append(color_by)

        work = _aggregate(work, group_cols, y_cols, aggregate)

    sort_cols = [color_by, x_plot] if color_by else [x_plot]
    work = work.sort_values(sort_cols)

    if cumulative:
        if color_by:
            for col in y_cols:
                work[col] = work.groupby(color_by, dropna=False)[col].cumsum()
        else:
            for col in y_cols:
                work[col] = work[col].cumsum()

    data = []

    groups = (
        work.groupby(color_by, dropna=False)
        if color_by
        else [(None, work)]
    )

    for group_value, g in groups:
        for y_col in y_cols:
            name = y_col if group_value is None else str(group_value)

            if group_value is not None and len(y_cols) > 1:
                name = f"{group_value} - {y_col}"

            data.append({
                "type": "scatter",
                "mode": "lines+markers",
                "x": g[x_plot].tolist(),
                "y": g[y_col].tolist(),
                "name": name,
            })

    return {
        "data": data,
        "layout": {
            "title": {"text": title or f"{', '.join(y_cols)} over {x}"},
            "xaxis": {"title": {"text": x}},
            "yaxis": {"title": {"text": ", ".join(y_cols)}},
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


def plot_pie_chart(
    df: pd.DataFrame,
    labels: str,
    values: str,
    *,
    aggregate: str | None = None,
    group_by: list[str] | None = None,
    title: str | None = None,
    hole: float = 0.0,
    template: str = "plotly_white",
) -> dict:
    """
    Create a Plotly pie chart.

    Use only for clear part-to-whole, share, contribution,
    percentage, or composition questions.
    """

    _validate_columns(df, [labels, values])

    work = df.copy()

    if aggregate is not None:
        group_cols = list(group_by or [labels])

        if labels not in group_cols:
            group_cols.insert(0, labels)

        work = _aggregate(work, group_cols, [values], aggregate)

    return {
        "data": [
            {
                "type": "pie",
                "labels": work[labels].astype(str).tolist(),
                "values": work[values].tolist(),
                "hole": hole,
            }
        ],
        "layout": {
            "title": {"text": title or f"{values} share by {labels}"},
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


def plot_scatter_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    color_by: str | None = None,
    size_by: str | None = None,
    text_by: str | None = None,
    title: str | None = None,
    template: str = "plotly_white",
) -> dict:
    """
    Create a Plotly scatter chart.

    Use for numeric-vs-numeric relationships, correlations,
    crossplots, clusters, or row-level comparisons.
    """

    y_cols = _as_list(y)

    required = [x, *y_cols]
    if color_by:
        required.append(color_by)
    if size_by:
        required.append(size_by)
    if text_by:
        required.append(text_by)

    _validate_columns(df, required)

    work = df.copy()
    data = []

    groups = (
        work.groupby(color_by, dropna=False)
        if color_by
        else [(None, work)]
    )

    for group_value, g in groups:
        for y_col in y_cols:
            name = y_col if group_value is None else str(group_value)

            if group_value is not None and len(y_cols) > 1:
                name = f"{group_value} - {y_col}"

            trace = {
                "type": "scatter",
                "mode": "markers",
                "x": g[x].tolist(),
                "y": g[y_col].tolist(),
                "name": name,
            }

            if size_by:
                size_values = pd.to_numeric(g[size_by], errors="coerce").fillna(0)
                max_size = max(float(size_values.max()), 1.0)

                trace["marker"] = {
                    "size": size_values.tolist(),
                    "sizemode": "area",
                    "sizeref": max_size / 40,
                    "sizemin": 4,
                }

            if text_by:
                trace["text"] = g[text_by].astype(str).tolist()
                trace["hovertemplate"] = (
                    f"{x}: %{{x}}<br>"
                    f"{y_col}: %{{y}}<br>"
                    f"{text_by}: %{{text}}"
                    "<extra></extra>"
                )

            data.append(trace)

    return {
        "data": data,
        "layout": {
            "title": {"text": title or f"{', '.join(y_cols)} vs {x}"},
            "xaxis": {"title": {"text": x}},
            "yaxis": {"title": {"text": ", ".join(y_cols)}},
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


# ============================================================
# Prompt
# ============================================================

CHART_TOOL_SELECTION_PROMPT = """
You are a chart presentation planner.

Given:
- user query
- table summaries
- column names, roles, cardinality, and descriptions

Select the most appropriate charting tool only when:
- Required columns clearly exist
- The analytical intent is clear
- Required parameters can be assigned confidently
- The chart will meaningfully answer the query

AVAILABLE TOOLS
- plot_bar_chart
- plot_line_chart
- plot_pie_chart
- plot_scatter_chart

TOOL RULES
- Use plot_bar_chart for quantitative values or counts compared across categorical or bucketed temporal dimensions.
- Use plot_line_chart for trends, time series, ordered progression, or cumulative values over time.
- Use plot_pie_chart only for clear part-to-whole/share/composition questions where categories form one meaningful total.
- Use plot_scatter_chart for numeric-vs-numeric relationships, correlations, crossplots, clusters, or row-level comparisons.

PARAMETER RULES
- Use sum aggregation by default for additive quantities unless the query specifies another aggregation.
- Use color_by only when a clear secondary categorical grouping is requested.
- Use cumulative=true only when the query explicitly asks for cumulative/running total.
- Use date_bucket only when the query asks or implies monthly, quarterly, yearly, weekly, or daily grouping.
- Never rely on unsupported assumptions.
- If any core parameter is ambiguous, return tool=null.

OUTPUT
Return only valid JSON:

{
  "tool": "<tool_name_or_null>",
  "reason": "<brief reason>",
  "args": {} | null
}

**IMPORTANT**:
Do not invent arguments. Only use the exact arguments listed below.

plot_bar_chart args:
x, y, aggregate, group_by, color_by, orientation, barmode, title, template

plot_line_chart args:
x, y, aggregate, group_by, color_by, date_bucket, cumulative, title, template

plot_pie_chart args:
labels, values, aggregate, group_by, title, hole, template

plot_scatter_chart args:
x, y, color_by, size_by, text_by, title, template

Never use unsupported arguments such as facet_by, facet, row, col, animation, or transform.


"""


# ============================================================
# Optional LLM invoke helper
# ============================================================

def select_chart_tool(
    llm,
    user_query: str,
    table_context: str,
) -> dict:
    """
    Calls the LLM to select one charting tool and its arguments.
    """

    from langchain_core.messages import SystemMessage, HumanMessage

    messages = [
        SystemMessage(content=CHART_TOOL_SELECTION_PROMPT),
        HumanMessage(content=f"""
USER QUERY
{user_query}

TABLES
{table_context}
""")
    ]

    response = llm.invoke(messages)

    text = response.content.strip()

    if text.startswith("```"):
        text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    return json.loads(text)


def run_selected_chart_tool(
    tool_selection: dict,
    df: pd.DataFrame,
) -> dict | None:
    """
    Executes the selected chart tool against a dataframe.
    """

    tool = tool_selection.get("tool")
    args = tool_selection.get("args") or {}

    if tool is None or tool == "null":
        return None

    registry = {
        "create_combined_category_column": create_combined_category_column,
        "plot_bar_chart": plot_bar_chart,
        "plot_line_chart": plot_line_chart,
        "plot_pie_chart": plot_pie_chart,
        "plot_scatter_chart": plot_scatter_chart,
    }

    if tool not in registry:
        raise ValueError(f"Unknown chart tool: {tool}")

    return registry[tool](df=df, **args)

def run_chart_plan(
    plan: dict,
    df: pd.DataFrame,
) -> dict | None:
    """
    plan shape:
    {
  "steps": [
    {
      "tool": "create_combined_category_column",
      "args": {
        "col1": "sector",
        "col2": "continent",
        "new_col": "sector_continent",
        "sep": " / "
      }
    },
    {
      "tool": "plot_bar_chart",
      "args": {
        "x": "sector_continent",
        "y": "oil_production",
        "aggregate": "sum",
        "group_by": ["sector_continent"],
        "title": "Oil Production by Sector and Continent"
      }
    }
  ]
    }
    """

    work = df.copy()
    figure = None

    for step in plan["steps"]:
        tool = step["tool"]
        args = step.get("args") or {}

        if tool == "create_combined_category_column":
            work = create_combined_category_column(work, **args)

        elif tool == "plot_bar_chart":
            figure = plot_bar_chart(work, **args)

        elif tool == "plot_line_chart":
            figure = plot_line_chart(work, **args)

        elif tool == "plot_pie_chart":
            figure = plot_pie_chart(work, **args)

        elif tool == "plot_scatter_chart":
            figure = plot_scatter_chart(work, **args)

        else:
            raise ValueError(f"Unknown tool: {tool}")

    return figure

# ============================================================
# Example usage
# ============================================================

p = TableResponseProcessor() 
table_context = p.process([(table, df)])
#
tool_selection = select_chart_tool(
     llm=llm,
     user_query=user_query,
     table_context=table_context,
 )
#
fig = run_selected_chart_tool(tool_selection, df)
#
import plotly.io as pio
pio.show(fig)

: 

In [ ]:
from __future__ import annotations

import json
import re
from typing import Any

import numpy as np
import pandas as pd


# ============================================================
# Shared helpers
# ============================================================

ALLOWED_AGGS = {"sum", "mean", "median", "min", "max", "count", "nunique"}


def _as_list(value):
    if value is None:
        return []
    return [value] if isinstance(value, str) else list(value)


def _validate_columns(df: pd.DataFrame, columns: list[str], label: str = "column"):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f"Missing {label}(s): {missing}")


def _aggregate(
    df: pd.DataFrame,
    group_by: list[str],
    value_cols: list[str],
    aggregate: str,
) -> pd.DataFrame:
    if aggregate not in ALLOWED_AGGS:
        raise ValueError(f"Unsupported aggregate: {aggregate}")

    _validate_columns(df, group_by, "group_by column")
    _validate_columns(df, value_cols, "value column")

    return (
        df.groupby(group_by, dropna=False, as_index=False)[value_cols]
        .agg(aggregate)
    )


def _bucket_date(
    df: pd.DataFrame,
    date_col: str,
    bucket: str,
) -> tuple[pd.DataFrame, str]:
    out = df.copy()
    bucket_col = f"{date_col}_{bucket}"

    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

    if bucket == "D":
        out[bucket_col] = out[date_col].dt.to_period("D").dt.to_timestamp()
    elif bucket == "W":
        out[bucket_col] = out[date_col].dt.to_period("W").dt.start_time
    elif bucket == "M":
        out[bucket_col] = out[date_col].dt.to_period("M").dt.to_timestamp()
    elif bucket == "Q":
        out[bucket_col] = out[date_col].dt.to_period("Q").dt.to_timestamp()
    elif bucket == "Y":
        out[bucket_col] = out[date_col].dt.to_period("Y").dt.to_timestamp()
    else:
        raise ValueError("date_bucket must be one of: D, W, M, Q, Y")

    return out, bucket_col


def _strip_markdown_json(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def _filter_args(tool: str, args: dict[str, Any]) -> dict[str, Any]:
    allowed_args = {
        "create_combined_category_column": {
            "col1", "col2", "new_col", "sep"
        },
        "plot_bar_chart": {
            "x", "y", "aggregate", "group_by", "color_by",
            "orientation", "barmode", "title", "template"
        },
        "plot_line_chart": {
            "x", "y", "aggregate", "group_by", "color_by",
            "date_bucket", "cumulative", "title", "template"
        },
        "plot_pie_chart": {
            "labels", "values", "aggregate", "group_by",
            "title", "hole", "template"
        },
        "plot_scatter_chart": {
            "x", "y", "color_by", "size_by", "text_by",
            "title", "template"
        },
    }

    if tool not in allowed_args:
        raise ValueError(f"Unknown tool: {tool}")

    return {k: v for k, v in args.items() if k in allowed_args[tool]}


# ============================================================
# Data-prep tool
# ============================================================

def create_combined_category_column(
    df: pd.DataFrame,
    col1: str,
    col2: str,
    new_col: str | None = None,
    sep: str = " / ",
) -> pd.DataFrame:
    """
    Create a readable combined categorical column from two categorical columns.
    """
    _validate_columns(df, [col1, col2])

    out = df.copy()
    new_col = new_col or f"{col1}_{col2}"

    out[new_col] = (
        out[col1].fillna("").astype(str)
        + sep
        + out[col2].fillna("").astype(str)
    )

    return out


# ============================================================
# Chart tools
# ============================================================

def plot_bar_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    aggregate: str | None = None,
    group_by: list[str] | None = None,
    color_by: str | None = None,
    orientation: str = "v",
    barmode: str = "group",
    title: str | None = None,
    template: str = "plotly_white",
) -> dict:
    y_cols = _as_list(y)

    required = [x, *y_cols]
    if color_by:
        required.append(color_by)

    _validate_columns(df, required)

    work = df.copy()

    if aggregate is not None:
        group_cols = _as_list(group_by) or [x]

        if x not in group_cols:
            group_cols.insert(0, x)

        if color_by and color_by not in group_cols:
            group_cols.append(color_by)

        work = _aggregate(work, group_cols, y_cols, aggregate)

    data = []

    groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

    for group_value, g in groups:
        for y_col in y_cols:
            name = y_col if group_value is None else str(group_value)

            if group_value is not None and len(y_cols) > 1:
                name = f"{group_value} - {y_col}"

            trace = {
                "type": "bar",
                "name": name,
                "orientation": orientation,
            }

            if orientation == "h":
                trace["x"] = g[y_col].tolist()
                trace["y"] = g[x].astype(str).tolist()
            else:
                trace["x"] = g[x].astype(str).tolist()
                trace["y"] = g[y_col].tolist()

            data.append(trace)

    return {
        "data": data,
        "layout": {
            "title": {"text": title or f"{', '.join(y_cols)} by {x}"},
            "xaxis": {"title": {"text": ", ".join(y_cols) if orientation == "h" else x}},
            "yaxis": {"title": {"text": x if orientation == "h" else ", ".join(y_cols)}},
            "barmode": barmode,
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


def plot_line_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    aggregate: str | None = None,
    group_by: list[str] | None = None,
    color_by: str | None = None,
    date_bucket: str | None = None,
    cumulative: bool = False,
    title: str | None = None,
    template: str = "plotly_white",
) -> dict:
    y_cols = _as_list(y)

    required = [x, *y_cols]
    if color_by:
        required.append(color_by)

    _validate_columns(df, required)

    work = df.copy()
    x_plot = x

    if date_bucket is not None:
        work, x_plot = _bucket_date(work, x, date_bucket)

    if aggregate is not None:
        group_cols = _as_list(group_by) or [x]

        if x_plot not in group_cols:
            group_cols.insert(0, x_plot)

        if color_by and color_by not in group_cols:
            group_cols.append(color_by)

        work = _aggregate(work, group_cols, y_cols, aggregate)

    sort_cols = [color_by, x_plot] if color_by else [x_plot]
    work = work.sort_values(sort_cols)

    if cumulative:
        if color_by:
            for col in y_cols:
                work[col] = work.groupby(color_by, dropna=False)[col].cumsum()
        else:
            for col in y_cols:
                work[col] = work[col].cumsum()

    data = []
    groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

    for group_value, g in groups:
        for y_col in y_cols:
            name = y_col if group_value is None else str(group_value)

            if group_value is not None and len(y_cols) > 1:
                name = f"{group_value} - {y_col}"

            data.append({
                "type": "scatter",
                "mode": "lines+markers",
                "x": g[x_plot].tolist(),
                "y": g[y_col].tolist(),
                "name": name,
            })

    return {
        "data": data,
        "layout": {
            "title": {"text": title or f"{', '.join(y_cols)} over {x}"},
            "xaxis": {"title": {"text": x}},
            "yaxis": {"title": {"text": ", ".join(y_cols)}},
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


def plot_pie_chart(
    df: pd.DataFrame,
    labels: str,
    values: str,
    *,
    aggregate: str | None = None,
    group_by: list[str] | None = None,
    title: str | None = None,
    hole: float = 0.0,
    template: str = "plotly_white",
) -> dict:
    _validate_columns(df, [labels, values])

    work = df.copy()

    if aggregate is not None:
        group_cols = list(group_by or [labels])

        if labels not in group_cols:
            group_cols.insert(0, labels)

        work = _aggregate(work, group_cols, [values], aggregate)

    return {
        "data": [
            {
                "type": "pie",
                "labels": work[labels].astype(str).tolist(),
                "values": work[values].tolist(),
                "hole": hole,
            }
        ],
        "layout": {
            "title": {"text": title or f"{values} share by {labels}"},
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


def plot_scatter_chart(
    df: pd.DataFrame,
    x: str,
    y: str | list[str],
    *,
    color_by: str | None = None,
    size_by: str | None = None,
    text_by: str | None = None,
    title: str | None = None,
    template: str = "plotly_white",
) -> dict:
    y_cols = _as_list(y)

    required = [x, *y_cols]
    if color_by:
        required.append(color_by)
    if size_by:
        required.append(size_by)
    if text_by:
        required.append(text_by)

    _validate_columns(df, required)

    work = df.copy()
    data = []

    groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

    for group_value, g in groups:
        for y_col in y_cols:
            name = y_col if group_value is None else str(group_value)

            if group_value is not None and len(y_cols) > 1:
                name = f"{group_value} - {y_col}"

            trace = {
                "type": "scatter",
                "mode": "markers",
                "x": g[x].tolist(),
                "y": g[y_col].tolist(),
                "name": name,
            }

            if size_by:
                size_values = pd.to_numeric(g[size_by], errors="coerce").fillna(0)
                max_size = max(float(size_values.max()), 1.0)

                trace["marker"] = {
                    "size": size_values.tolist(),
                    "sizemode": "area",
                    "sizeref": max_size / 40,
                    "sizemin": 4,
                }

            if text_by:
                trace["text"] = g[text_by].astype(str).tolist()
                trace["hovertemplate"] = (
                    f"{x}: %{{x}}<br>"
                    f"{y_col}: %{{y}}<br>"
                    f"{text_by}: %{{text}}"
                    "<extra></extra>"
                )

            data.append(trace)

    return {
        "data": data,
        "layout": {
            "title": {"text": title or f"{', '.join(y_cols)} vs {x}"},
            "xaxis": {"title": {"text": x}},
            "yaxis": {"title": {"text": ", ".join(y_cols)}},
            "template": template,
        },
        "config": {
            "responsive": True,
            "displaylogo": False,
        },
    }


# ============================================================
# Prompt
# ============================================================

CHART_TOOL_SELECTION_PROMPT = """
You are a chart presentation planner.

Given:
- user query
- table summaries
- column names, roles, cardinality, and descriptions

Return a minimal chart plan.

The plan may contain:
- zero or one preparation step
- exactly one final plotting step

AVAILABLE PREPARATION TOOL
- create_combined_category_column

Use create_combined_category_column only when a chart needs one readable category made from two categorical columns.

AVAILABLE PLOTTING TOOLS
- plot_bar_chart
- plot_line_chart
- plot_pie_chart
- plot_scatter_chart

TOOL RULES
- Use plot_bar_chart for quantitative values or counts compared across categorical or bucketed temporal dimensions.
- Use plot_line_chart for trends, time series, ordered progression, or cumulative values over time.
- Use plot_pie_chart only for clear part-to-whole/share/composition questions where categories form one meaningful total.
- Use plot_scatter_chart for numeric-vs-numeric relationships, correlations, crossplots, clusters, or row-level comparisons.

PARAMETER RULES
- Required columns must clearly exist.
- Required parameters must be assigned confidently.
- Use sum aggregation by default for additive quantities unless the query specifies another aggregation.
- Use color_by only when a clear secondary categorical grouping is requested.
- Use cumulative=true only when the query explicitly asks for cumulative/running total.
- Use date_bucket only when the query asks or implies monthly, quarterly, yearly, weekly, or daily grouping.
- If multiple categorical dimensions are required but direct charting would be unclear, create a combined category first.
- Never rely on unsupported assumptions.
- If no chart is clearly suitable, return steps as [].

ARGUMENT RULES
Do not invent arguments. Only use the exact arguments listed below.

create_combined_category_column args:
col1, col2, new_col, sep

plot_bar_chart args:
x, y, aggregate, group_by, color_by, orientation, barmode, title, template

plot_line_chart args:
x, y, aggregate, group_by, color_by, date_bucket, cumulative, title, template

plot_pie_chart args:
labels, values, aggregate, group_by, title, hole, template

plot_scatter_chart args:
x, y, color_by, size_by, text_by, title, template

Never use unsupported arguments such as facet_by, facet, row, col, animation, transform, filters, where, or sort.

OUTPUT
Return only valid JSON in this shape:

{
  "reason": "<brief reason>",
  "steps": [
    {
      "tool": "<tool_name>",
      "args": {}
    }
  ]
}
"""

CHART_AGENT_PROMPT = """
You are a chart planning agent.

You receive:
- user query
- table summaries
- column names, roles, cardinality, and descriptions

Return a JSON plan with:
- optional preprocess step
- exactly one plot step

PREPROCESS TOOL

preprocess_for_chart:
Use only when a needed chart column can be derived safely.

Args:
{
  "create_combined_category": null | {
    "col1": "<categorical_col>",
    "col2": "<categorical_col>",
    "new_col": "<new_col>",
    "sep": " / "
  },
  "create_date_bucket": null | {
    "date_col": "<date_col>",
    "bucket": "D|W|M|Q|Y",
    "new_col": "<new_col>"
  }
}

PLOT TOOLS

plot_bar_chart:
Use for quantitative values or counts compared across categorical or bucketed temporal dimensions.
Args:
{
  "x": "<category_or_bucket_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>"],
  "color_by": null | "<secondary_category_col>",
  "orientation": "v|h",
  "barmode": "group|stack|relative",
  "title": "<title>"
}

plot_line_chart:
Use for trends, time series, ordered progression, or cumulative values.
Args:
{
  "x": "<time_or_ordered_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>"],
  "color_by": null | "<category_col>",
  "date_bucket": null | "D|W|M|Q|Y",
  "cumulative": true|false,
  "title": "<title>"
}

plot_pie_chart:
Use only for part-to-whole/share/composition questions.
Args:
{
  "labels": "<category_col>",
  "values": "<numeric_col>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<label_col>"],
  "hole": 0.0,
  "title": "<title>"
}

plot_scatter_chart:
Use for numeric-vs-numeric relationships, correlations, crossplots, clusters, or row-level comparisons.
Args:
{
  "x": "<numeric_col>",
  "y": "<numeric_col_or_list>",
  "color_by": null | "<category_col>",
  "size_by": null | "<numeric_col>",
  "text_by": null | "<label_col>",
  "title": "<title>"
}

RULES
- Return only valid JSON.
- Do not invent tools.
- Do not invent arguments.
- Use only columns that exist or are created by preprocess_for_chart.
- Prefer no preprocess when existing columns are sufficient.
- Use sum by default for additive quantities unless otherwise specified.
- If uncertain, return {"reason": "...", "preprocess": null, "plot": null}.

OUTPUT SHAPE
{
  "reason": "<brief reason>",
  "preprocess": null | {
    "tool": "preprocess_for_chart",
    "args": {}
  },
  "plot": null | {
    "tool": "<plot_tool>",
    "args": {}
  }
}
"""

# ============================================================
# LLM + executor
# ============================================================

def select_chart_plan(
    llm,
    user_query: str,
    table_context: str,
) -> dict:
    from langchain_core.messages import SystemMessage, HumanMessage

    messages = [
        SystemMessage(content=CHART_TOOL_SELECTION_PROMPT),
        HumanMessage(content=f"""
USER QUERY
{user_query}

TABLES
{table_context}
"""),
    ]

    response = llm.invoke(messages)
    text = _strip_markdown_json(response.content)
    return json.loads(text)


def normalize_old_tool_selection(selection: dict) -> dict:
    """
    Allows old shape:
      {"tool": "...", "args": {...}}
    to still work by converting it to:
      {"reason": "...", "steps": [{"tool": "...", "args": {...}}]}
    """
    if "steps" in selection:
        return selection

    tool = selection.get("tool")
    args = selection.get("args") or {}

    if tool is None or tool == "null":
        return {"reason": selection.get("reason", ""), "steps": []}

    return {
        "reason": selection.get("reason", ""),
        "steps": [{"tool": tool, "args": args}],
    }


def run_chart_plan(
    plan: dict,
    df: pd.DataFrame,
) -> dict | None:
    plan = normalize_old_tool_selection(plan)

    steps = plan.get("steps") or []
    if not steps:
        return None

    work = df.copy()
    figure = None

    plotting_tools = {
        "plot_bar_chart": plot_bar_chart,
        "plot_line_chart": plot_line_chart,
        "plot_pie_chart": plot_pie_chart,
        "plot_scatter_chart": plot_scatter_chart,
    }

    for i, step in enumerate(steps):
        tool = step.get("tool")
        args = step.get("args") or {}
        args = _filter_args(tool, args)

        is_last = i == len(steps) - 1

        if tool == "create_combined_category_column":
            if is_last:
                raise ValueError("Final step cannot be create_combined_category_column.")
            work = create_combined_category_column(work, **args)

        elif tool in plotting_tools:
            if not is_last:
                raise ValueError("Plotting tool must be the final step.")
            figure = plotting_tools[tool](df=work, **args)

        else:
            raise ValueError(f"Unknown tool: {tool}")

    return figure


# Backwards-compatible aliases
select_chart_tool = select_chart_plan
run_selected_chart_tool = run_chart_plan


# ============================================================
# Example usage
# ============================================================

p = TableResponseProcessor()
table_context = p.process([(table, df)])
#
plan = select_chart_plan(
     llm=llm,
     user_query=user_query,
     table_context=table_context,
)
#
print(json.dumps(plan, indent=2))

fig = run_chart_plan(plan, df)
#
import plotly.io as pio
pio.show(fig)

: 

In [ ]:
user_query = response['structured_response'].user_query
table_name = "quarterly_water_injection_by_subzone_year"
table = single_agent_data.catalog_snapshot( table_name ).derived_tables[0]
df = single_agent_data.get_table_as_df(table_name)
df


: 

: 

: 

: 

In [ ]:
from __future__ import annotations

from typing import Any, Iterable
import pandas as pd

import copy
import json
import pandas as pd


PLOTLY_ARRAY_KEYS = {
    "x",
    "y",
    "z",
    "r",
    "theta",
    "values",
    "labels",
    "text",
    "hovertext",
    "customdata",
    "ids",
    "parents",
}


def resolve_plotly_figure(
    figure: dict,
    df: pd.DataFrame,
    *,
    to_json: bool = False,
) -> dict | str:
    """
    Replace column-name placeholders in Plotly traces with real DataFrame values.

    Example:
        "x": "DATE"              -> "x": [...]
        "y": "oil_production"    -> "y": [...]
        "labels": "producer"     -> "labels": [...]
        "values": "oil_rate"     -> "values": [...]
    """

    resolved = copy.deepcopy(figure)

    for trace in resolved.get("data", []):
        for key in PLOTLY_ARRAY_KEYS:
            value = trace.get(key)

            if isinstance(value, str) and value in df.columns:
                trace[key] = df[value].tolist()

    if to_json:
        return json.dumps(resolved, default=str)

    return resolved

class TableResponseProcessor:
    """
    Builds compact, chart-oriented table summaries for a Plotly presenter LLM.

    The LLM receives metadata and limited column examples, but never the full data.
    """

    def __init__(
        self,
        max_examples: int = 3,
        low_cardinality_threshold: int = 10,
        medium_cardinality_threshold: int = 50,
        categorical_numeric_threshold: int = 12,
    ):
        self.max_examples = max_examples
        self.low_cardinality_threshold = low_cardinality_threshold
        self.medium_cardinality_threshold = medium_cardinality_threshold
        self.categorical_numeric_threshold = categorical_numeric_threshold

    def process(
        self,
        items: Iterable[tuple["TableCard", pd.DataFrame]],
    ) -> str:
        """
        Process multiple (TableCard, DataFrame) pairs into one text block
        for the presenter prompt.
        """

        blocks: list[str] = []

        for table_card, df in items:
            blocks.append(self.process_table(df=df, table_card=table_card))

        #return "\n\n" + ("=" * 80) + "\n\n".join(blocks)
        return ("\n\n" + "=" * 80 + "\n\n").join(blocks)

    def process_table(
        self,
        df: pd.DataFrame,
        table_card: "TableCard",
    ) -> str:
        """
        Convert one DataFrame + TableCard into compact, LLM-friendly text.
        """

        table_name = getattr(table_card, "name", "unknown_table")
        table_description = getattr(table_card, "description", None)

        lines: list[str] = []

        lines.append(f"Table: {table_name}")

        if table_description:
            lines.append(f"Description: {table_description}")

        lines.append(f"Rows: {len(df)}")
        lines.append(f"Columns: {len(df.columns)}")

        lines.append("")
        lines.append("Column summaries:")

        for col in df.columns:
            summary = self.infer_column_summary(df[col])

            description = self.get_column_description(table_card, col)
            if description:
                summary["description"] = description

            lines.append(f"- Column: {col}")

            preferred_order = [
                "description",
                "dtype",
                "role",
                "unique_count",
                "unique_ratio",
                "cardinality",
                "null_count",
                "null_ratio",
                "min",
                "max",
                "mean",
                "median",
                "min_date",
                "max_date",
                "common_interval",
                "is_monotonic_increasing",
                "has_negative_values",
                "has_zero_values",
                "example_values",
            ]

            for key in preferred_order:
                if key in summary:
                    lines.append(f"  {key}: {summary[key]}")

        return "\n".join(lines)

    def infer_column_role(self, s: pd.Series) -> str:
        """
        Infer a chart-oriented semantic role.
        """

        if pd.api.types.is_datetime64_any_dtype(s):
            return "temporal"

        if pd.api.types.is_bool_dtype(s):
            return "categorical"

        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)

            if nunique <= self.categorical_numeric_threshold:
                return "categorical_numeric"

            return "quantitative"

        if pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s):
            parsed = pd.to_datetime(s.dropna(), errors="coerce")
            valid_ratio = parsed.notna().mean() if len(parsed) else 0.0

            if valid_ratio >= 0.9:
                return "temporal"

            return "categorical"

        return "unknown"

    def infer_column_summary(self, s: pd.Series) -> dict[str, Any]:
        """
        Produce compact metadata for one DataFrame column.
        """

        non_null = s.dropna()

        row_count = len(s)
        non_null_count = len(non_null)
        null_count = row_count - non_null_count
        null_ratio = null_count / row_count if row_count else 0.0

        unique_count = non_null.nunique(dropna=True)
        unique_ratio = unique_count / row_count if row_count else 0.0

        role = self.infer_column_role(s)

        summary: dict[str, Any] = {
            "dtype": str(s.dtype),
            "role": role,
            "non_null_count": int(non_null_count),
            "null_count": int(null_count),
            "null_ratio": round(null_ratio, 4),
            "unique_count": int(unique_count),
            "unique_ratio": round(unique_ratio, 4),
        }

        if role in {"categorical", "categorical_numeric"}:
            summary["cardinality"] = self.classify_cardinality(unique_count)

            examples = (
                non_null
                .drop_duplicates()
                .astype(str)
                .head(self.max_examples)
                .tolist()
            )

            if examples:
                summary["example_values"] = examples

        elif role == "quantitative":
            numeric = pd.to_numeric(non_null, errors="coerce").dropna()

            if not numeric.empty:
                summary.update(
                    {
                        "min": round(float(numeric.min()), 4),
                        "max": round(float(numeric.max()), 4),
                        "mean": round(float(numeric.mean()), 4),
                        "median": round(float(numeric.median()), 4),
                        "has_negative_values": bool((numeric < 0).any()),
                        "has_zero_values": bool((numeric == 0).any()),
                    }
                )

        elif role == "temporal":
            dt = pd.to_datetime(non_null, errors="coerce").dropna()

            if not dt.empty:
                summary.update(
                    {
                        "min_date": str(dt.min()),
                        "max_date": str(dt.max()),
                        "is_monotonic_increasing": bool(dt.is_monotonic_increasing),
                    }
                )

                if len(dt) > 1:
                    diffs = dt.sort_values().diff().dropna()
                    if not diffs.empty:
                        mode_diff = diffs.mode()
                        if not mode_diff.empty:
                            summary["common_interval"] = str(mode_diff.iloc[0])

        return summary

    def classify_cardinality(self, unique_count: int) -> str:
        """
        Convert unique count into low/medium/high cardinality label.
        """

        if unique_count <= self.low_cardinality_threshold:
            return "low"

        if unique_count <= self.medium_cardinality_threshold:
            return "medium"

        return "high"

    def get_column_description(
        self,
        table_card: "TableCard",
        column_name: str,
    ) -> str | None:
        """
        Extract column description from a TableCard, if present.

        Supports columns represented as either objects or dictionaries.
        """

        columns = getattr(table_card, "columns", None)

        if not columns:
            return None

        for col in columns:
            if isinstance(col, dict):
                name = col.get("name")
                description = col.get("description")
            else:
                name = getattr(col, "name", None)
                description = getattr(col, "description", None)

            if name == column_name:
                return description

        return None
    


    

: 

In [ ]:
PLOTLY_PRESENTER_PROMPT1 = """
You are a Plotly.js figure JSON generator.

You are given:
- a user query
- one table summary
- the table name
- column names, column roles, and column descriptions when available

Your job is to return only a Plotly.js figure JSON object that answers the user query.

The output will be used as:

Plotly.newPlot(element, figure.data, figure.layout, figure.config)

USER QUERY
{user_query}

TABLE
{tables}

STRICT OUTPUT RULES
- Return only valid JSON.
- Do not return markdown.
- Do not explain anything.
- Do not include comments.
- Do not include extra top-level fields.
- The top-level JSON object must contain only: data, layout, config.
- Use only valid Plotly figure keys.
- Do not use custom keys like table, column, split_by, placeholder, encoding, or fields.
- Do not use real data values.
- Use only columns that exist in the table summary.
- Do not invent column names.
- Wherever Plotly expects an array of data values, use the column name as a string placeholder.

PLACEHOLDER RULE
Use only the raw column name as the placeholder.

Examples:
"x": "DATE"
"y": "oil_production"
"text": "producer"
"labels": "producer"
"values": "oil_production"

CHART SELECTION RULES
- temporal + quantitative -> scatter lines+markers
- categorical + quantitative -> bar
- ranked/top/bottom -> bar, horizontal if labels are long
- share/percentage/fraction/contribution of total -> pie, only for low-cardinality categories
- two quantitative columns row-by-row -> scatter markers
- one numeric value -> indicator
- distribution/frequency/spread of one numeric column -> histogram
- temporal + several numeric columns in wide table -> one line trace per numeric column
- temporal + category + quantity in long table -> one line trace only; put category in text/hovertext
- several metrics with same unit -> multiple traces
- metrics with different units -> use y2 only if clearly needed
- use heatmap only for x-category + y-category + numeric-value matrix style data
- never use Plotly transforms
- never aggregate, sort, filter, group, or calculate unless already present in the table

STYLE RULES
- Include a clear title.
- Include axis titles when the chart has x/y axes.
- Use hovertemplate when useful.
- Set config.responsive to true.
- Set config.displaylogo to false.

RETURN SHAPE
Return only JSON shaped like this:

{{
  "data": [
    {{
      "type": "scatter",
      "mode": "lines+markers",
      "x": "<column_name>",
      "y": "<column_name>"
    }}
  ],
  "layout": {{
    "title": {{
      "text": "<chart title>"
    }},
  }},
  "config": {{
    "responsive": true,
    "displaylogo": false
  }}
}}
"""

: 

In [ ]:
PLOTLY_PRESENTER_PROMPT2 = """
You are a Plotly.js figure JSON generator.

You are given:
- a user query
- one table summary
- the table name
- column names, column roles, and column descriptions when available

===============================================================================
YOUR RESPONSIBILITIES
===============================================================================
- Analyze the user intent and the information provided in the table. 
- Decide HOW best display the information to a human.
- ALWAYS output your reasoning steps.
- Consider:
    1. What variable or variables must be in the X axis
    2. Determine if the variable in the X axis is Categorical (Nominal, ordinal), numerical or time?
    3. What variable(s) must be in the Y axis
    4. Is this a single-trace plot?
    5. From the user intent, determine if grouping is needed.  
    6. You must print a  Plotly.js figure JSON object that answers the user query.
 

===============================================================================
STRICT plotly JSON RULES
===============================================================================
- enclode the json plotly figure inside <plot> and </plot> 
- Do not include extra top-level fields.
- NO TRANSFORMS: Never use the transforms property.
- GROUPING VIA TRACES: If grouping is needed (e.g., by Subzone), create a separate trace object {{ "type": "bar", "name": "GROUP_NAME", ... } for each unique group.
- The top-level JSON object must contain only: data, layout, config.
- Use only valid Plotly figure keys.
- Do not use custom keys like table, column, split_by, placeholder, encoding, or fields.
- Do not use real data values.
- USE ONLY COLUMNS THAT EXIST in the table summary.
- Do not invent column names.
- Wherever Plotly expects an array of data values, use the column name as a string placeholder.

===============================================================================
PLACEHOLDER RULE
===============================================================================
Use only the raw column name as the placeholder.

Examples:
"x": "DATE"
"y": "oil_production"
"text": "producer"
"labels": "producer"
"values": "oil_production"

    

USER QUERY
{user_query}

TABLE
{tables}

""" 

k = """
===============================================================================
YOUR RESPONSIBILITIES
===============================================================================

- Do NOT recompute or modify numeric values.
- Do NOT add new analytics or invent new fields.
- Produce UI-ready JSON only.
- Your must return Plotly.js figure JSON object that answers the user query.




USER QUERY
{user_query}

TABLE
{tables}


PLACEHOLDER RULE
Use only the raw column name as the placeholder.

Examples:
"x": "DATE"
"y": "oil_production"
"text": "producer"
"labels": "producer"
"values": "oil_production"


STYLE RULES
- Include a clear title.
- Include axis titles when the chart has x/y axes.
- Use hovertemplate when useful.
- Set config.responsive to true.
- Set config.displaylogo to false.



RETURN SHAPE
Return only JSON shaped like this:

{{
  "data": [...],
  "layout": {{
    
    "title": {{
      "text": "<chart title>"
    }},
        "colorway": ["#4A90E2", "#50E3C2"],

        "xaxis": {{
          "title": {{ "text": "X-axis title" }}
        }},

        "yaxis": {{
          "title": {{ "text": "Y-axis title" }}
        }},

  }},
  "config": {{
    "responsive": true,
    "displaylogo": false
  }}
}}
"""

: 

: 

: 

In [ ]:
VEGA_LITE_PROMPT = """
You are a Vega-Lite (v5) chart specification generator.

You are given:
- A user query.
- A table summary (table name, column names, and types).

===============================================================================
VISUALIZATION LOGIC
===============================================================================
1. Time Series: Use `mark: "line"` or `"area"`.
2. Grouped Comparisons: Use `mark: "bar"`. To group bars side-by-side, use the `xOffset` encoding.
3. Part-to-Whole: Use `mark: "arc"`.
4. Relationships: Use `mark: "point"` for scatter plots.

===============================================================================
TRANSFORM & DATA RULES
===============================================================================
- COLUMN CONCATENATION: If you need to combine columns (e.g., Year and Quarter), use a `calculate` transform:
  "transform": [{{"calculate": "datum.year + ' Q' + datum.quarter", "as": "period"}}]
- SORTING: Always ensure time-based axes are sorted chronologically using the `sort` property in the encoding.
- NO PLOTS WITH RAW DATA: Use the placeholder "TABLE_DATA" for the `values` key in the `data` block.
- DATA FORMAT: Assume the data is a flat list of dictionaries (one per row).

===============================================================================
STRICT VEGA-LITE JSON RULES
===============================================================================
- Enclose the JSON inside <plot> and </plot> tags.
- Use only valid Vega-Lite v5 properties.
- Do NOT invent column names.
- Always include a `title` and clear `axis` labels.

===============================================================================
INTERACTIVITY & ENHANCEMENT RULES (MANDATORY)
===============================================================================
- ZOOM & PAN: Always include the following params block at the top level:
  "params": [{{"name": "grid", "select": "interval", "bind": "scales"}}]
- TOOLTIPS: Always include a "tooltip" array in the encoding block containing all relevant columns.
- COLUMN CONCATENATION: If you need to combine columns (e.g., Year and Quarter), use a `calculate` transform:
  "transform": [{{"calculate": "datum.YEAR + ' Q' + datum.quarter", "as": "period"}}]
- SORTING: Always sort time-based axes chronologically using the `sort` property.


===============================================================================
OUTPUT FORMAT EXAMPLE
===============================================================================
Reasoning: [Briefly explain chart choice and any transforms used]
<plot>
{{
  "$schema": "https://github.io",
  "data": {{ "values": "TABLE_DATA" }},
  "transform": [ ... ],
  "mark": "bar",
  "encoding": {{
    "x": {{ "field": "period", "type": "nominal" }},
    "y": {{ "field": "total_water_injection_volume", "type": "quantitative" }},
    "xOffset": {{ "field": "SUBZONE" }},
    "color": {{ "field": "SUBZONE" }}
  }}
}}
</plot>

USER QUERY: {user_query}
TABLE: {tables}
"""


: 

In [ ]:

class TableResponseProcessor:
    """
    Builds compact, chart-oriented table summaries for a Plotly presenter LLM.

    The LLM receives metadata and limited column examples, but never the full data.
    """

    def __init__(
        self,
        max_examples: int = 3,
        low_cardinality_threshold: int = 10,
        medium_cardinality_threshold: int = 50,
        categorical_numeric_threshold: int = 12,
    ):
        self.max_examples = max_examples
        self.low_cardinality_threshold = low_cardinality_threshold
        self.medium_cardinality_threshold = medium_cardinality_threshold
        self.categorical_numeric_threshold = categorical_numeric_threshold

    def process(
        self,
        items: Iterable[tuple["TableCard", pd.DataFrame]],
    ) -> str:
        """
        Process multiple (TableCard, DataFrame) pairs into one text block
        for the presenter prompt.
        """

        blocks: list[str] = []

        for table_card, df in items:
            blocks.append(self.process_table(df=df, table_card=table_card))

        #return "\n\n" + ("=" * 80) + "\n\n".join(blocks)
        return ("\n\n" + "=" * 80 + "\n\n").join(blocks)

    def process_table(
        self,
        df: pd.DataFrame,
        table_card: "TableCard",
    ) -> str:
        """
        Convert one DataFrame + TableCard into compact, LLM-friendly text.
        """

        table_name = getattr(table_card, "name", "unknown_table")
        table_description = getattr(table_card, "description", None)

        lines: list[str] = []

        lines.append(f"Table: {table_name}")

        if table_description:
            lines.append(f"Description: {table_description}")

        lines.append(f"Rows: {len(df)}")
        lines.append(f"Columns: {len(df.columns)}")

        lines.append("")
        lines.append("Column summaries:")

        for col in df.columns:
            summary = self.infer_column_summary(df[col])

            description = self.get_column_description(table_card, col)
            if description:
                summary["description"] = description

            lines.append(f"- Column: {col}")

            preferred_order = [
                "description",
                "dtype",
                "role",
                "unique_count",
                "unique_ratio",
                "cardinality",
                "null_count",
                "null_ratio",
                "min",
                "max",
                "mean",
                "median",
                "min_date",
                "max_date",
                "common_interval",
                "is_monotonic_increasing",
                "has_negative_values",
                "has_zero_values",
                "example_values",
            ]

            for key in preferred_order:
                if key in summary:
                    lines.append(f"  {key}: {summary[key]}")

        return "\n".join(lines)

    def infer_column_role(self, s: pd.Series) -> str:
        """
        Infer a chart-oriented semantic role.
        """

        if pd.api.types.is_datetime64_any_dtype(s):
            return "temporal"

        if pd.api.types.is_bool_dtype(s):
            return "categorical"

        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)

            if nunique <= self.categorical_numeric_threshold:
                return "categorical_numeric"

            return "quantitative"

        if pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s):
            parsed = pd.to_datetime(s.dropna(), errors="coerce")
            valid_ratio = parsed.notna().mean() if len(parsed) else 0.0

            if valid_ratio >= 0.9:
                return "temporal"

            return "categorical"

        return "unknown"

    def infer_column_summary(self, s: pd.Series) -> dict[str, Any]:
        """
        Produce compact metadata for one DataFrame column.
        """

        non_null = s.dropna()

        row_count = len(s)
        non_null_count = len(non_null)
        null_count = row_count - non_null_count
        null_ratio = null_count / row_count if row_count else 0.0

        unique_count = non_null.nunique(dropna=True)
        unique_ratio = unique_count / row_count if row_count else 0.0

        role = self.infer_column_role(s)

        summary: dict[str, Any] = {
            "dtype": str(s.dtype),
            "role": role,
            "non_null_count": int(non_null_count),
            "null_count": int(null_count),
            "null_ratio": round(null_ratio, 4),
            "unique_count": int(unique_count),
            "unique_ratio": round(unique_ratio, 4),
        }

        if role in {"categorical", "categorical_numeric"}:
            summary["cardinality"] = self.classify_cardinality(unique_count)

            examples = (
                non_null
                .drop_duplicates()
                .astype(str)
                .head(self.max_examples)
                .tolist()
            )

            if examples:
                summary["example_values"] = examples

        elif role == "quantitative":
            numeric = pd.to_numeric(non_null, errors="coerce").dropna()

            if not numeric.empty:
                summary.update(
                    {
                        "min": round(float(numeric.min()), 4),
                        "max": round(float(numeric.max()), 4),
                        "mean": round(float(numeric.mean()), 4),
                        "median": round(float(numeric.median()), 4),
                        "has_negative_values": bool((numeric < 0).any()),
                        "has_zero_values": bool((numeric == 0).any()),
                    }
                )

        elif role == "temporal":
            dt = pd.to_datetime(non_null, errors="coerce").dropna()

            if not dt.empty:
                summary.update(
                    {
                        "min_date": str(dt.min()),
                        "max_date": str(dt.max()),
                        "is_monotonic_increasing": bool(dt.is_monotonic_increasing),
                    }
                )

                if len(dt) > 1:
                    diffs = dt.sort_values().diff().dropna()
                    if not diffs.empty:
                        mode_diff = diffs.mode()
                        if not mode_diff.empty:
                            summary["common_interval"] = str(mode_diff.iloc[0])

        return summary

    def classify_cardinality(self, unique_count: int) -> str:
        """
        Convert unique count into low/medium/high cardinality label.
        """

        if unique_count <= self.low_cardinality_threshold:
            return "low"

        if unique_count <= self.medium_cardinality_threshold:
            return "medium"

        return "high"

    def get_column_description(
        self,
        table_card: "TableCard",
        column_name: str,
    ) -> str | None:
        """
        Extract column description from a TableCard, if present.

        Supports columns represented as either objects or dictionaries.
        """

        columns = getattr(table_card, "columns", None)

        if not columns:
            return None

        for col in columns:
            if isinstance(col, dict):
                name = col.get("name")
                description = col.get("description")
            else:
                name = getattr(col, "name", None)
                description = getattr(col, "description", None)

            if name == column_name:
                return description

        return None
    


: 

In [ ]:

user_query = response['structured_response'].user_query
table_name = "quarterly_water_injection_by_subzone_year"
table = single_agent_data.catalog_snapshot( table_name ).derived_tables[0]
df = single_agent_data.get_table_as_df(table_name)
df

: 

In [ ]:
import re 

p = TableResponseProcessor()

table_context = p.process( [(table,df)] )  

prompt = VEGA_LITE_PROMPT .format(
    tables = table_context,
    user_query = user_query
)




messages = [
    {'role':'system', 'content': prompt }
]
viz_response = llm.invoke(messages)



: 

In [ ]:
viz_response.pretty_print()


: 

: 

: 

In [ ]:

import re
text = viz_response.content.strip()
text  = re.search(r'<plot>(.*?)</plot>', text, re.DOTALL)
##text  = re.search(r'```json(.*?)```', text, re.DOTALL)
text = text.group(1).strip()
  
#text = text.split("```json")[1].replace("```","")

print(text)


: 

In [ ]:
import json
import altair as alt

# 1. Your JSON from the agent (as a string)

agent_output = """
{
  "$schema": "https://vega.github.io/schema/vega-lite/v6.json",
  "data": { "values": "TABLE_DATA" },

  "width": 800,
  "height": 420,

  "params": [
    {
      "name": "zoom_pan",
      "select": {
        "type": "interval"
      },
      "bind": "scales"
    },
    {
      "name": "subzone_toggle",
      "select": {
        "type": "point",
        "fields": ["SUBZONE"]
      },
      "bind": "legend"
    },
    {
      "name": "brush",
      "select": {
        "type": "interval",
        "encodings": ["x"]
      }
    },
    {
      "name": "hover",
      "select": {
        "type": "point",
        "on": "mouseover",
        "clear": "mouseout"
      }
    }
  ],

  "transform": [
    {
      "calculate": "toString(datum.YEAR) + ' Q' + toString(datum.quarter)",
      "as": "period"
    }
  ],

  "mark": {
    "type": "bar",
    "cursor": "pointer",
    "stroke": "black",
    "strokeWidth": 0.5
  },

  "encoding": {
    "x": {
      "field": "period",
      "type": "nominal",
      "title": "Quarter",
      "sort": null,
      "axis": {
        "labelAngle": 0
      }
    },

    "y": {
      "field": "total_water_injection_volume",
      "type": "quantitative",
      "title": "Total Water Injection Volume"
    },

    "xOffset": {
      "field": "SUBZONE",
      "type": "nominal"
    },

    "color": {
      "field": "SUBZONE",
      "type": "nominal",
      "title": "Subzone"
    },

    "opacity": {
      "condition": [
        {
          "param": "hover",
          "empty": false,
          "value": 1
        },
        {
          "param": "subzone_toggle",
          "value": 0.9
        },
        {
          "param": "brush",
          "empty": false,
          "value": 0.8
        }
      ],
      "value": 0.2
    },

    "tooltip": [
      {
        "field": "period",
        "type": "nominal",
        "title": "Quarter"
      },
      {
        "field": "SUBZONE",
        "type": "nominal",
        "title": "Subzone"
      },
      {
        "field": "total_water_injection_volume",
        "type": "quantitative",
        "title": "Water Injection Volume",
        "format": ",.2f"
      },
      {
        "field": "YEAR",
        "type": "ordinal",
        "title": "Year"
      },
      {
        "field": "quarter",
        "type": "ordinal",
        "title": "Quarter Number"
      }
    ]
  },

  "title": "Total Water Injection Volume by Quarter and Subzone"
}
"""

agent_output1 = """
{
  "$schema": "https://vega.github.io/schema/vega-lite/v6.json",
  "data": { "values": "TABLE_DATA" },

  "width": 400,
  "height": 320,

  "params": [
    {
      "name": "zoom_pan",
      "select": {
        "type": "interval"
      },
      "bind": "scales"
    },
    {
      "name": "subzone_toggle",
      "select": {
        "type": "point",
        "fields": ["SUBZONE"]
      },
      "bind": "legend"
    },
    {
      "name": "brush",
      "select": {
        "type": "interval",
        "encodings": ["x"]
      }
    },
    {
      "name": "hover",
      "select": {
        "type": "point",
        "on": "mouseover",
        "clear": "mouseout"
      }
    }
  ],

  "transform": [
    {
      "calculate": "datetime(datum.YEAR, (datum.quarter - 1) * 3, 1)",
      "as": "period_date"
    },
    {
      "calculate": "toString(datum.YEAR) + ' Q' + toString(datum.quarter)",
      "as": "period_label"
    }
  ],

  "mark": {
    "type": "bar",
    "cursor": "pointer",
    "stroke": "black",
    "strokeWidth": 0.5
  },

  "encoding": {
    "x": {
      "field": "period_date",
      "type": "temporal",
      "title": "Quarter",
      "axis": {
        "format": "%Y Q%q",
        "labelAngle": 0
      }
    },

    "y": {
      "field": "total_water_injection_volume",
      "type": "quantitative",
      "title": "Total Water Injection Volume"
    },

    "xOffset": {
      "field": "SUBZONE",
      "type": "nominal"
    },

    "color": {
      "field": "SUBZONE",
      "type": "nominal",
      "title": "Subzone"
    },

    "opacity": {
      "condition": [
        {
          "param": "hover",
          "empty": false,
          "value": 1
        },
        {
          "param": "subzone_toggle",
          "value": 0.9
        },
        {
          "param": "brush",
          "empty": false,
          "value": 0.8
        }
      ],
      "value": 0.2
    },

    "tooltip": [
      {
        "field": "period_label",
        "type": "nominal",
        "title": "Quarter"
      },
      {
        "field": "SUBZONE",
        "type": "nominal",
        "title": "Subzone"
      },
      {
        "field": "total_water_injection_volume",
        "type": "quantitative",
        "title": "Water Injection Volume",
        "format": ",.2f"
      }
    ]
  },

  "title": "Total Water Injection Volume by Quarter and Subzone"
}
"""


#agent_output = text 
# 2. Parse the string into a dictionary
chart_spec = json.loads(text)

# 3. Swap the placeholder for your actual dataframe data
chart_spec['data']['values'] = df.to_dict(orient='records')

# 4. Display the chart
alt.Chart.from_dict(chart_spec).display()


: 

In [ ]:
chart = alt.Chart.from_dict(chart_spec).interactive()
chart.display()

chart.data


: 

In [ ]:
data_from_chart = pd.DataFrame(chart_spec['data']['values'])
data_from_chart

: 

In [ ]:
%pip install altair


: 

In [ ]:

#text = viz_response.content.strip()
#text = re.sub(r"^```json\s*|\s*```$", "", text, flags=re.DOTALL)
figure = json.loads(text)


#figure['layout']["template"]="plotly_dark"


resolved_figure = resolve_plotly_figure(figure, df)
import plotly.io as pio
pio.show(resolved_figure)

: 

In [ ]:
df

: 

In [ ]:
list(pio.templates)

: 

In [ ]:
c=""" 
CHART SELECTION RULES

First identify the main analytical intent of the user query:

1. Trend over an ordered variable
- Use a scatter trace with mode "lines+markers".
- Use this when the query asks for evolution, trend, history, change over time, monthly, yearly, daily, cumulative, forecast, or progression.
- x should be the temporal or ordered column.
- y should be the main quantitative column.
- Do not use pie charts for trends.

2. Category comparison / ranking
- Use a bar trace.
- Use this when the query asks to compare, rank, top N, bottom N, highest, lowest, by category, by well, by sector, by producer, by injector, etc.
- x should usually be the categorical column.
- y should be the quantitative column.
- If category labels are likely long, use a horizontal bar:
  - "type": "bar"
  - "orientation": "h"
  - x = quantitative column
  - y = categorical column

3. Part-to-whole / share
- Use a pie trace only when the query asks for share, percentage, fraction, contribution, mix, breakdown, distribution of a total, or proportion.
- Use pie only if the categorical column has low cardinality.
- For pie charts:
  - labels = categorical column
  - values = quantitative column
- Prefer a bar chart instead of pie if there are many categories, if ranking matters, or if categories do not represent parts of one total.

4. Numeric relationship / correlation
- Use a scatter trace with mode "markers".
- Use this when the query asks for relationship, correlation, vs, versus, dependency, association, crossplot, or compare two numeric variables row-by-row.
- x should be one quantitative column.
- y should be another quantitative column.
- Use text or hovertext for an identifier column if available.

5. Single value / one-row result
- If the table has one row and one main numeric value, use an indicator trace.
- If the table has one row and several numeric values, use a bar chart.
- If the user asks for a simple KPI/value, prefer an indicator.

6. Distribution of one numeric column
- If the query asks for distribution, histogram, frequency, spread, or variability, use a histogram trace.
- x should be the numeric column.

7. Multiple quantitative columns
- If the query asks to compare several metrics with the same unit, use multiple traces.
- If the metrics have different units, prefer separate y-axes only when necessary:
  - first metric uses "yaxis": "y"
  - second metric uses "yaxis": "y2"
  - layout must include yaxis and yaxis2
- Avoid dual axes unless the query clearly needs it.

8. Temporal + category + quantity
- If there is a temporal column, a categorical column with low/medium cardinality, and a quantitative column:
  - Use one scatter line trace only if the frontend will not split traces.
  - Put the category column in "text" or "hovertext".
  - Do not invent custom "split_by".
- If the input table is already wide, with one temporal column and several numeric series columns, create one trace per numeric series.

9. Heatmap / matrix-shaped tables
- Use heatmap only if the table clearly represents a matrix or has x-category, y-category, and numeric value columns.
- Use:
  - "type": "heatmap"
  - "x": categorical or temporal x column
  - "y": categorical y column
  - "z": numeric value column
- Do not use heatmap unless the table summary supports it.

General defaults:
- Prefer the simplest chart that answers the query.
- If the user explicitly asks for a chart type, follow it unless it conflicts with the data.
- If no clear chart type is implied:
  - temporal + quantitative -> line chart
  - categorical + quantitative -> bar chart
  - two quantitative columns -> scatter chart
  - categorical share of quantitative total -> pie chart
- Do not use Plotly transforms.
- Do not aggregate, calculate, sort, filter, or group in the figure JSON unless the result already exists in the provided table.

"""


len(c)/4,int(len(c.split()) * 1.3)

: 

In [ ]:
figure = json.loads(response.content)
import plotly.io as pio

pio.renderers.default = "notebook_connected"

fig = {
    "data": figure["data"],
    "layout": figure.get("layout", {}),
    "config": figure.get("config", {}),
}

pio.show(fig)

: 

: 

In [ ]:


presenter_agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=presenter_system_prompt,
        checkpointer=None,
    )

: 

: 

: 

In [ ]:
r = response['structured_response']
tables = r.tables 
query = r.user_query 
table_names  = [ t.table_name for t in tables ]
descriptions = [t.description for t in tables ] 

print( query )
print( table_names[0] )
print( descriptions[0] )
single_agent_data.get_table_as_df( table_names[0])

#viz_prompt """
#
#"""

print('-----')
single_agent_data.catalog_snapshot('largest_yoy_injection_drop')#.derived_tables# get_tables_brief_description()

: 

: 

In [ ]:
r = response['structured_response']
query = r.user_query 
tables = r.tables 
table_names  = [ t.table_name for t in tables ]
descriptions = [t.description for t in tables ] 

kinds = [] 
for t in tables:
    if t.row_count < 2 : kinds.append( 'text' )

single_agent_data.get_table_as_df( table_names[0])
TableItemAgentResponse


: 

In [ ]:
print(single_agnet_tools[0].func('largest_yoy_injection_drop_per_subzone'))#.catalog_snapshot()



: 

In [ ]:
#single_agent_data.catalog_snapshot().derived_tables#'largest_yoy_injection_drop_per_subzone')

print( response['structured_response'].tables )


: 

In [ ]:
table_name = response['structured_response'].tables[0].table_name
print(table_name )
single_agent_data.get_table_as_df( table_name )



: 

In [ ]:
data._catalog.snapshot()#. catalog_snapshot().derived_tables# 'largest_yoy_injection_drop_per_subzone' )

: 

# As a graph, tool or node. 

In [ ]:
from langgraph.graph import StateGraph, END

def executor_prompt_builder_from_structured_plan(
    idiom: str,
    idiom_rules: dict,
    structured_plan,
    user_query: str = "" 
) -> str:

    step_blocks = []
    for step in structured_plan.steps:

        source_tables = ", ".join(step.source_tables)
        reusable_tables = (
            ", ".join(step.reusable_tables)
            if step.reusable_tables
            else "None"
        )

        block = f"""
        Step {step.step_id}
        Target Table: {step.target_table}
        Source Tables: {source_tables}
        Reusable Tables: {reusable_tables}
        Logic: {step.logic}
        """.strip()

        step_blocks.append(block)
        
    formatted_plan = "\n\n".join(step_blocks)
    idiom_examples = "\n".join(
            [f"- {k}: {v}" for k, v in idiom_rules.items()]
        )
 
    # -----------------------------------------
    # Build final system prompt
    # -----------------------------------------
    prompt = system_prompt_sql_executor_template.format(
        idiom=idiom,
        idiom_examples=idiom_examples,
        plan=formatted_plan
    )

    if user_query:
        prompt = prompt + f"\n\n**USER QUERY**:\n{user_query}\n"
        
    return prompt

def planner_prompt_builder( tools:SmartDataTools )->str:
    #txt = data.catalog_snapshot()
    #txt = json.dumps( data.catalog_snapshot(), indent=3)
    txt = tools.catalog_snapshot()
    prompt = system_prompt_sql_planner_template.format(catalog=txt)

    return prompt 

class DataAnalystState(BaseModel):
    user_query: str
    refined_query: Optional[str] = None  
    plan: Optional[ExecutionPlan] = Field(
        default=None,
        description="Structured execution plan generated by planner",
    )

    execution_result: Optional[AgentTableResponse]  = Field(
        default=None,
        description="Executor output",
    )

    error: Optional[str] = Field(
        default=None,
        description="Execution or planning error",
    )

class DataAnalyst:

    def __init__(self, llm, tables_dict: None | Dict[str,pd.DataFrame] = None, 
                            known_table_models: None | Dict[str,TableCard] = None ):

        self._data  : SmartData #= SmartData()
        self._tools : List[StructuredTool] #= SmartDataTools( self._data ).get_tools()
        self._llm = llm 
        self._idiom = 'duckdb'
        self._graph = None 

    
        self._idiom_rules= {'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
        'date truncation': "Use DATE_TRUNC('month', column).",
        'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
        'string concatenation': 'Use the || operator or CONCAT().',
        'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
        'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
        'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)'
        }

        if tables_dict and known_table_models:
            self.initialize_from_known_tables( tables_dict, known_table_models)


    def get_result_as_dataframes(
        self,
        state: DataAnalystState | dict,
    ) -> Dict[str, pd.DataFrame]:
        """
        Retrieve all materialized result tables as DataFrames.

        Supports:
        - DataAnalystState
        - raw graph dict output
        """
        # -----------------------------------------------------
        # Normalize state
        # -----------------------------------------------------
        if isinstance(state, dict):
            state = DataAnalystState(**state)

        results = {}

        if state.execution_result is None:
            return results

        if not state.execution_result.tables:
            return results

        # -----------------------------------------------------
        # Load tables
        # -----------------------------------------------------
        for table_item in state.execution_result.tables:
            table_name = table_item.table_name

            try:
                results[table_name] = self._data.get_table_as_df(
                    table_name
                )

            except Exception as e:
                print(
                    f"Failed loading table '{table_name}': {str(e)}"
                )

        return results

    def initialize_from_known_tables( self, 
                                     tables_dict: Dict[str,pd.DataFrame], 
                                     known_table_models: Dict[str,TableCard] ):
   
        self._data = SmartData() 
        self._data.initialize_from_named_dataframes( tables_dict, known_table_models)
        self._smart_data_tools = SmartDataTools( self._data )#.get_tools()
        self._tools = self._smart_data_tools.get_tools() 
        
    def planner_node( self, state:DataAnalystState):
        
        # generate structured plan
        messages = [] 
        instruction = state.user_query
        planner_prompt = planner_prompt_builder(self._smart_data_tools)
        print(planner_prompt)
        llm = self._llm
        try: 
            messages = [{
                "role": "system",
                "content": planner_prompt
                },
                {
                "role": "user",
                "content": instruction
                }]

            # Structured planner
            structured_llm = llm.with_structured_output(ExecutionPlan)
            plan = structured_llm.invoke(messages)
            #print( plan )


            return state.model_copy(
                update={
                    "plan": plan,
                    #"error": None,
                })


            #return {
            #    #"catalog": catalog,
            #    "plan": plan#.dict() if hasattr(plan, "dict") else plan,
            #}
        
        except Exception as e:
            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })

  
    def execution_node( self, state:DataAnalystState):
  
        llm = self._llm
        plan = state.plan
        idiom= self._idiom
        idiom_rules = self._idiom_rules

        q = plan.user_query if plan.refined_query is None else  plan.refined_query

        executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan,q )

        print("\n" + "=" * 80)
        print("EXECUTOR PROMPT")
        print("=" * 80)
        print(executor_prompt)
        print("=" * 80 + "\n")
        

        agent = create_agent(
                model=llm,
                system_prompt=executor_prompt,
                tools=self._tools,
                response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )
        
        try:
            response = run_agent_stream_values(agent, {} ) 
        

            #print('******************************')
            #print(response)
            #print('******************************')
            


            #return {
            ##"catalog": catalog,
            #"execution_result": response#.dict() if hasattr(plan, "dict") else plan,
            #} 
        
            return state.model_copy(
            update={
                "execution_result": response['structured_response'],
                "error": None,
            })

        
        except Exception as e:
            print("STREAM FAILED:", str(e))


            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })



            #return {
            #"catalog": catalog,
            #"error": str(e)#.dict() if hasattr(plan, "dict") else plan,
            #} 
            
        
    # Conditional after planner
    def should_continue_node(self, state: DataAnalystState):
        if state.error:
            return END

        if state.plan is None:
            return END

        if hasattr(state.plan, "steps") and not state.plan.steps:
            return END

        return "execution"

    # ------------------------------------------------------------------
    # Graph builder
    # ------------------------------------------------------------------
    def as_graph(self):
        builder = StateGraph(DataAnalystState)

        builder.add_node("planner", self.planner_node)
        builder.add_node("execution", self.execution_node)

        builder.set_entry_point("planner")

        builder.add_conditional_edges(
            "planner",
            self.should_continue_node,
            {
                "execution": "execution",
                END: END,
            },
        )

        builder.add_edge("execution", END)

        self._graph = builder.compile()

        return self._graph

    # ------------------------------------------------------------------
    # Main runner
    # ------------------------------------------------------------------
    def run(self, user_query: str):
        """
        Execute full planner -> executor workflow.

        Parameters
        ----------
        user_query : str
            Natural language analytical request.

        Returns
        -------
        DataAnalystState
            Final validated workflow state.
        """
        if self._graph is None:
            self.as_graph()

        initial_state = DataAnalystState(
            user_query=user_query,
            refined_query=None,
            plan=None,
            execution_result=None,
            error=None,
        )

        result = self._graph.invoke(initial_state)
        return self.normalize_state(result)

        # LangGraph may return dict depending on version
        #if isinstance(result, dict):
        #    return DataAnalystState(**result)
        #return result

# ------------------------------------------------------------------
# Normalize output
# ------------------------------------------------------------------
    def normalize_state(self, result) -> DataAnalystState:
        """
        Convert graph output into validated DataAnalystState.
        """
        if isinstance(result, DataAnalystState):
            return result

        if isinstance(result, dict):
            return DataAnalystState(**result)

        raise TypeError(
            f"Unsupported graph output type: {type(result)}"
        )


    # ------------------------------------------------------------------
    # Direct graph invoke wrapper
    # ------------------------------------------------------------------
    def invoke_graph(self, user_query: str) -> DataAnalystState:
        """
        Direct graph call but always returns structured state.
        """
        if self._graph is None:
            self.as_graph()

        result = self._graph.invoke(
            DataAnalystState(
                user_query=user_query,
                refined_query=None,
                plan=None,
                execution_result=None,
                error=None,
            )
        )

        return self.normalize_state(result)


    # ------------------------------------------------------------------
    # Export as LangChain tool
    # ------------------------------------------------------------------
    def as_tool(self) -> StructuredTool:
        """
        Expose DataAnalyst as a reusable tool for other agents.
        """

        def _run_analysis(user_query: str) -> dict:
            state = self.run(user_query)
            return state 
        
            return {
                "plan": (
                    state.plan.model_dump()
                    if state.plan else None
                ),
                "execution_result": (
                    state.execution_result.model_dump()
                    if state.execution_result else None
                ),
                "error": state.error,
            }

        return StructuredTool.from_function(
            func=_run_analysis,
            name="data_analyst",
            description=(
                "Executes structured analytical workflows over known tabular datasets. "
                "Useful for SQL-style table generation, ranking, aggregations, "
                "time-series analysis, and derived table creation."
            ),
        )


: 

In [ ]:
analyst = DataAnalyst( llm, df_dict, known_table_models )


: 

In [ ]:

tool = analyst.as_tool()
print(tool)

# =========================================================
# Direct tool invocation
# =========================================================
tool_result = tool.invoke(
    {
        "user_query": (
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    }
)
print("TOOL RESULT:")
type(tool_result)
tool_result.execution_result.tables

: 

In [ ]:
dfs = analyst.get_result_as_dataframes(tool_result)
print(dfs.keys())
first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)



: 

: 

In [ ]:

graph = analyst.as_graph() 
display( graph )

# =========================================================
# Direct graph invoke
# =========================================================
result = graph.invoke(
    DataAnalystState(
        user_query=(
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    )
)

# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)





: 

In [ ]:
result
analyst.# .get_result_as_dataframes()#'injector_wells_ranked_by_water_injection')

: 

In [ ]:
dir(analyst)

: 

In [ ]:



# =========================================================
# Simple invoke test
# =========================================================
response = analyst.run(
    "Create two separate tables: "
    "one ranking injector wells by total water injection volume, "
    "and another ranking producer wells by total oil production volume."
)

print("FINAL RESPONSE:")
# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)

: 

# End 


: 

: 

In [ ]:

llm = azure_llm_if()



planner_prompt = planner_prompt_builder(smart_data_tools)
print(planner_prompt)


: 

In [ ]:
messages = [] 
instruction = query11


messages = [{
    "role": "system",
    "content": planner_prompt
    },
    {
    "role": "user",
    "content": instruction
    }]


plan = llm.invoke(messages).content

print( plan )

: 

In [ ]:


idiom = 'duckdb'
idiom_rules= {'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
 'date truncation': "Use DATE_TRUNC('month', column).",
 'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
 'string concatenation': 'Use the || operator or CONCAT().',
 'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
 'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
 'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)'
}
executor_prompt = executor_prompt_builder(idiom, idiom_rules,plan)
agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        #response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 

: 

In [ ]:
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))



: 

# Add structure to the plan 


In [ ]:


# generate structured plan
messages = [] 
instruction = query11


messages = [{
    "role": "system",
    "content": planner_prompt
    },
    {
    "role": "user",
    "content": instruction
    }]

# Structured planner
structured_llm = llm.with_structured_output(ExecutionPlan)
plan = structured_llm.invoke(messages)
print( plan )






: 

In [ ]:
def executor_prompt_builder_from_structured_plan(
    idiom: str,
    idiom_rules: dict,
    structured_plan,
    user_query: str = "" 
) -> str:

    step_blocks = []
    for step in structured_plan.steps:

        source_tables = ", ".join(step.source_tables)
        reusable_tables = (
            ", ".join(step.reusable_tables)
            if step.reusable_tables
            else "None"
        )

        block = f"""
        Step {step.step_id}
        Target Table: {step.target_table}
        Source Tables: {source_tables}
        Reusable Tables: {reusable_tables}
        Logic: {step.logic}
        """.strip()

        step_blocks.append(block)
        
    formatted_plan = "\n\n".join(step_blocks)
    idiom_examples = "\n".join(
            [f"- {k}: {v}" for k, v in idiom_rules.items()]
        )
 
    # -----------------------------------------
    # Build final system prompt
    # -----------------------------------------
    prompt = system_prompt_sql_executor_template.format(
        idiom=idiom,
        idiom_examples=idiom_examples,
        plan=formatted_plan
    )

    if user_query:
        prompt = prompt + f"\n\nThis is the user query:\n{user_query}"
        
    return prompt


executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan)


print(executor_prompt)



: 

In [ ]:
executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan)


print(executor_prompt)

agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        #response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))





: 

: 

# Add structure to the output 

In [ ]:

llm = azure_llm_if()


: 

In [ ]:
class PlanStep(BaseModel):
    step_id: int
    target_table: str
    source_tables: List[str]
    reusable_tables: List[str] = Field(default_factory=list)
    logic: str

class ExecutionPlan(BaseModel):
    #raw_query: str 
    user_query: str
    tables_needed: List[str] = Field( default=[],description="List of all the tables in the catalog that will be needed to answer the question")
    steps: List[PlanStep]



class TableItemAgentResponse(BaseModel):
    table_name: str = Field(description="Name of a materialized output table")
    description: str = Field(description="Brief summary of the table contents")
        
   
      
class AgentTableResponse(BaseModel):
    # Literal ensures the LLM chooses only these specific strings
    agent: Literal["analyst"] = Field(
        default="analyst", 
        description="The role of the agent. Always 'analyst'."
    )
    tables: List[str] = Field(default=[], description="Comma-separated list of table names")

    #text : Optional[str]  = Field(default=None, description="textual response")
    #tables: List[TableItemAgentResponse] = Field(default_factory=list, description="List of materialized output tables")
    
    

: 

In [ ]:
agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))




: 

# Prompt-chaining implementation 

In [ ]:
from typing import TypedDict
from typing import TypedDict, Optional, Dict, Any, Callable
from pydantic import BaseModel, Field
from typing import List
import sys, pathlib, json, pprint, pandas as pd 
from pathlib import Path
from pydantic import Field,BaseModel 
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from runtime.v4.semantics.load_semantics import load_semantics
from runtime.v4.semantics.semantic_models import * 
from runtime.v4.analyst_agent.catalog import Catalog
from runtime.v4.analyst_agent.smart_data import SmartData
from runtime.v4.analyst_agent.smart_data_tools  import SmartDataTools


from langgraph.graph import StateGraph, END



class PlanStep(BaseModel):
    step_id: int
    target_table: str
    source_tables: List[str]
    reusable_tables: List[str] = Field(default_factory=list)
    logic: str

class ExecutionPlan(BaseModel):
    #raw_query: str 
    user_query: str
    refined_query: Optional[str] = None 
    tables_needed: List[str] = Field( default=[],description="List of all the tables in the catalog that will be needed to answer the question")
    steps: List[PlanStep]

class TableItemAgentResponse(BaseModel):
    table_name: str = Field(description="Name of a materialized output table")
    description: str = Field(description="Brief summary of the table contents")
        

class AgentTableResponse(BaseModel):
    # Literal ensures the LLM chooses only these specific strings
    agent: Literal["analyst"] = Field(
        default="analyst", 
        description="The role of the agent. Always 'analyst'."
    )
    user_query: str = Field( description='sanitized user query')
    tables: List[TableItemAgentResponse] = Field(default=[], description="Comma-separated list of table names")

    #text : Optional[str]  = Field(default=None, description="textual response")
    #tables: List[TableItemAgentResponse] = Field(default_factory=list, description="List of materialized output tables")
    
    

: 

In [ ]:



def executor_prompt_builder_from_structured_plan(
    idiom: str,
    idiom_rules: dict,
    structured_plan,
    user_query: str = "" 
) -> str:

    step_blocks = []
    for step in structured_plan.steps:

        source_tables = ", ".join(step.source_tables)
        reusable_tables = (
            ", ".join(step.reusable_tables)
            if step.reusable_tables
            else "None"
        )

        block = f"""
        Step {step.step_id}
        Target Table: {step.target_table}
        Source Tables: {source_tables}
        Reusable Tables: {reusable_tables}
        Logic: {step.logic}
        """.strip()

        step_blocks.append(block)
        
    formatted_plan = "\n\n".join(step_blocks)
    idiom_examples = "\n".join(
            [f"- {k}: {v}" for k, v in idiom_rules.items()]
        )
 
    # -----------------------------------------
    # Build final system prompt
    # -----------------------------------------
    prompt = system_prompt_sql_executor_template.format(
        idiom=idiom,
        idiom_examples=idiom_examples,
        plan=formatted_plan
    )

    if user_query:
        prompt = prompt + f"\n\n**USER QUERY**:\n{user_query}\n"
        
    return prompt

def planner_prompt_builder( tools:SmartDataTools )->str:
    #txt = data.catalog_snapshot()
    #txt = json.dumps( data.catalog_snapshot(), indent=3)
    txt = tools.catalog_snapshot()
    prompt = system_prompt_sql_planner_template.format(catalog=txt)

    return prompt 



class DataAnalystState(BaseModel):
    user_query: str
    refined_query: Optional[str] = None  
    plan: Optional[ExecutionPlan] = Field(
        default=None,
        description="Structured execution plan generated by planner",
    )

    execution_result: Optional[AgentTableResponse]  = Field(
        default=None,
        description="Executor output",
    )

    error: Optional[str] = Field(
        default=None,
        description="Execution or planning error",
    )

class DataAnalyst:

    def __init__(self, llm, tables_dict: None | Dict[str,pd.DataFrame] = None, 
                            known_table_models: None | Dict[str,TableCard] = None ):

        self._data  : SmartData #= SmartData()
        self._tools : List[StructuredTool] #= SmartDataTools( self._data ).get_tools()
        self._llm = llm 
        self._idiom = 'duckdb'
        self._graph = None 

    
        self._idiom_rules= {'date subtraction': "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
        'date truncation': "Use DATE_TRUNC('month', column).",
        'reserved keywords': 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
        'string concatenation': 'Use the || operator or CONCAT().',
        'boolean aggregation': 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.',
        'nested aggregates': 'Avoid nested aggregates—never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)',
        'cte helpers': 'Use CTEs to capture helper scalars (like current_year via MAX("DATE")) before performing group aggregations. For example:\n    WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX("DATE")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)'
        }

        if tables_dict and known_table_models:
            self.initialize_from_known_tables( tables_dict, known_table_models)


    def get_result_as_dataframes(
        self,
        state: DataAnalystState | dict,
    ) -> Dict[str, pd.DataFrame]:
        """
        Retrieve all materialized result tables as DataFrames.

        Supports:
        - DataAnalystState
        - raw graph dict output
        """
        # -----------------------------------------------------
        # Normalize state
        # -----------------------------------------------------
        if isinstance(state, dict):
            state = DataAnalystState(**state)

        results = {}

        if state.execution_result is None:
            return results

        if not state.execution_result.tables:
            return results

        # -----------------------------------------------------
        # Load tables
        # -----------------------------------------------------
        for table_item in state.execution_result.tables:
            table_name = table_item.table_name

            try:
                results[table_name] = self._data.get_table_as_df(
                    table_name
                )

            except Exception as e:
                print(
                    f"Failed loading table '{table_name}': {str(e)}"
                )

        return results

    def initialize_from_known_tables( self, 
                                     tables_dict: Dict[str,pd.DataFrame], 
                                     known_table_models: Dict[str,TableCard] ):
   
        self._data = SmartData() 
        self._data.initialize_from_named_dataframes( tables_dict, known_table_models)
        self._smart_data_tools = SmartDataTools( self._data )#.get_tools()
        self._tools = self._smart_data_tools.get_tools() 
        
    def planner_node( self, state:DataAnalystState):
        
        # generate structured plan
        messages = [] 
        instruction = state.user_query
        planner_prompt = planner_prompt_builder(self._smart_data_tools)
        print(planner_prompt)
        llm = self._llm
        try: 
            messages = [{
                "role": "system",
                "content": planner_prompt
                },
                {
                "role": "user",
                "content": instruction
                }]

            # Structured planner
            structured_llm = llm.with_structured_output(ExecutionPlan)
            plan = structured_llm.invoke(messages)
            #print( plan )


            return state.model_copy(
                update={
                    "plan": plan,
                    #"error": None,
                })


            #return {
            #    #"catalog": catalog,
            #    "plan": plan#.dict() if hasattr(plan, "dict") else plan,
            #}
        
        except Exception as e:
            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })

  
    def execution_node( self, state:DataAnalystState):
  
        llm = self._llm
        plan = state.plan
        idiom= self._idiom
        idiom_rules = self._idiom_rules

        q = plan.user_query if plan.refined_query is None else  plan.refined_query

        executor_prompt = executor_prompt_builder_from_structured_plan(idiom, idiom_rules,plan,q )

        print("\n" + "=" * 80)
        print("EXECUTOR PROMPT")
        print("=" * 80)
        print(executor_prompt)
        print("=" * 80 + "\n")
        

        agent = create_agent(
                model=llm,
                system_prompt=executor_prompt,
                tools=self._tools,
                response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )
        
        try:
            response = run_agent_stream_values(agent, {} ) 
        

            #print('******************************')
            #print(response)
            #print('******************************')
            


            #return {
            ##"catalog": catalog,
            #"execution_result": response#.dict() if hasattr(plan, "dict") else plan,
            #} 
        
            return state.model_copy(
            update={
                "execution_result": response['structured_response'],
                "error": None,
            })

        
        except Exception as e:
            print("STREAM FAILED:", str(e))


            return state.model_copy(
                update={
                    #"execution_result": response,
                    "error": str(e),
                })



            #return {
            #"catalog": catalog,
            #"error": str(e)#.dict() if hasattr(plan, "dict") else plan,
            #} 
            
        
    # Conditional after planner
    def should_continue_node(self, state: DataAnalystState):
        if state.error:
            return END

        if state.plan is None:
            return END

        if hasattr(state.plan, "steps") and not state.plan.steps:
            return END

        return "execution"

    # ------------------------------------------------------------------
    # Graph builder
    # ------------------------------------------------------------------
    def as_graph(self):
        builder = StateGraph(DataAnalystState)

        builder.add_node("planner", self.planner_node)
        builder.add_node("execution", self.execution_node)

        builder.set_entry_point("planner")

        builder.add_conditional_edges(
            "planner",
            self.should_continue_node,
            {
                "execution": "execution",
                END: END,
            },
        )

        builder.add_edge("execution", END)

        self._graph = builder.compile()

        return self._graph

    # ------------------------------------------------------------------
    # Main runner
    # ------------------------------------------------------------------
    def run(self, user_query: str):
        """
        Execute full planner -> executor workflow.

        Parameters
        ----------
        user_query : str
            Natural language analytical request.

        Returns
        -------
        DataAnalystState
            Final validated workflow state.
        """
        if self._graph is None:
            self.as_graph()

        initial_state = DataAnalystState(
            user_query=user_query,
            refined_query=None,
            plan=None,
            execution_result=None,
            error=None,
        )

        result = self._graph.invoke(initial_state)
        return self.normalize_state(result)

        # LangGraph may return dict depending on version
        #if isinstance(result, dict):
        #    return DataAnalystState(**result)
        #return result

# ------------------------------------------------------------------
# Normalize output
# ------------------------------------------------------------------
    def normalize_state(self, result) -> DataAnalystState:
        """
        Convert graph output into validated DataAnalystState.
        """
        if isinstance(result, DataAnalystState):
            return result

        if isinstance(result, dict):
            return DataAnalystState(**result)

        raise TypeError(
            f"Unsupported graph output type: {type(result)}"
        )


    # ------------------------------------------------------------------
    # Direct graph invoke wrapper
    # ------------------------------------------------------------------
    def invoke_graph(self, user_query: str) -> DataAnalystState:
        """
        Direct graph call but always returns structured state.
        """
        if self._graph is None:
            self.as_graph()

        result = self._graph.invoke(
            DataAnalystState(
                user_query=user_query,
                refined_query=None,
                plan=None,
                execution_result=None,
                error=None,
            )
        )

        return self.normalize_state(result)


    # ------------------------------------------------------------------
    # Export as LangChain tool
    # ------------------------------------------------------------------
    def as_tool(self) -> StructuredTool:
        """
        Expose DataAnalyst as a reusable tool for other agents.
        """

        def _run_analysis(user_query: str) -> dict:
            state = self.run(user_query)
            return state 
        
            return {
                "plan": (
                    state.plan.model_dump()
                    if state.plan else None
                ),
                "execution_result": (
                    state.execution_result.model_dump()
                    if state.execution_result else None
                ),
                "error": state.error,
            }

        return StructuredTool.from_function(
            func=_run_analysis,
            name="data_analyst",
            description=(
                "Executes structured analytical workflows over known tabular datasets. "
                "Useful for SQL-style table generation, ranking, aggregations, "
                "time-series analysis, and derived table creation."
            ),
        )




: 

In [ ]:
analyst = DataAnalyst( llm, df_dict, known_table_models )

tool = analyst.as_tool()
print(tool)


# =========================================================
# Direct tool invocation
# =========================================================
tool_result = tool.invoke(
    {
        "user_query": (
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    }
)
print("TOOL RESULT:")
type(tool_result)
tool_result.execution_result.tables





graph = analyst.as_graph() 
display( graph )

# =========================================================
# Direct graph invoke
# =========================================================
result = graph.invoke(
    DataAnalystState(
        user_query=(
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    )
)

# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]
display(first_df)




# =========================================================
# Simple invoke test
# =========================================================
response = analyst.run(
    "Create two separate tables: "
    "one ranking injector wells by total water injection volume, "
    "and another ranking producer wells by total oil production volume."
)

print("FINAL RESPONSE:")
#print(response)



: 

: 

: 

: 

In [ ]:

graph = analyst.as_graph() 
display( graph )

# Direct graph invoke
# =========================================================
result = graph.invoke(
    DataAnalystState(
        user_query=(
            "Create two separate tables: "
            "one ranking injector wells by total water injection volume, "
            "and another ranking producer wells by total oil production volume."
        )
    )
)

# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]

display(first_df)

: 

In [ ]:
# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(result)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]

display(first_df)

: 

In [ ]:
type(result)
result.keys()
type(result['execution_result'])
state = DataAnalystState(**result)
state

: 

: 

In [ ]:

# =========================================================
# Simple invoke test
# =========================================================
response = analyst.run(
    "Create two separate tables: "
    "one ranking injector wells by total water injection volume, "
    "and another ranking producer wells by total oil production volume."
)

print("FINAL RESPONSE:")
#print(response)



: 

In [ ]:
type(response)

: 

In [ ]:
# =========================================================
# Retrieve result tables as DataFrames
# =========================================================
dfs = analyst.get_result_as_dataframes(response)

print(dfs.keys())

first_key = list(dfs.keys())[0]
first_df = dfs[first_key]

display(first_df)


: 

In [ ]:
if response.plan:
    print("\nPLAN:")
    print(response.plan.model_dump_json(indent=3))


if response.execution_result:#" in response:
    print("\nEXECUTION RESULT:")
    print(response.execution_result.model_dump_json(indent=3))

: 

In [ ]:
response.execution_result.tables

: 

In [ ]:

executor_prompt = executor_prompt_builder(idiom, idiom_rules,plan)

agent = create_agent(
        model=llm,
        system_prompt=executor_prompt,
        tools=tools,
        #response_format=AgentTableResponse,
        #checkpointer= InMemorySaver() 
    )
 
try:
    response = run_agent_stream_values(agent, {} ) 
    last_response = response 
except Exception as e:
    print("STREAM FAILED:", str(e))


: 

In [ ]:
tools

: 

In [ ]:




class old:     

    def fdgdfgrun( self, user_query):

        llm = self._llm
        #planner
        planner_prompt = self._planner_prompt_builder()
        messages = [{
            "role": "system",
            "content": planner_prompt
            },
            {
            "role": "user",
            "content": user_query
            }]
        structured_llm = llm.with_structured_output(ExecutionPlan)

        plan = structured_llm.invoke(messages)
        print('Plan result')
        print( plan )
        
        executor_prompt = self._executor_prompt_builder(self._idiom, 
                                                        self._idiom_rules, 
                                                        plan )  
        executor_agent = create_agent(
                model=llm,
                system_prompt=executor_prompt,
                tools=self._tools,
                #response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )   
         
        

        #executor 
        try:
            response = run_agent_stream_values(executor_agent, {} ) 
            last_response = response 
        except Exception as e:
            print("STREAM FAILED:", str(e))





    def xxrun(self, user_query: str) -> Dict[str, Any]:

        initial_state: DataAnalystState = {
            "user_query": user_query,
            "plan": None,
            "execution_result": None,
            "error": None,
        }

        return self._graph.invoke(initial_state)    
    
    def planner_node(self,state: DataAnalystState) -> DataAnalystState:
        
        llm = self._llm
        
        # Structured planner
        structured_planner = llm.with_structured_output(ExecutionPlan)
        state["error"] = "" 
        try:
            planner_prompt = self._planner_prompt_builder()
            instruction = state['user_query']
            messages = [
                {
                    "role": "system",
                    "content": planner_prompt
                },
                {
                    "role": "user",
                    "content": instruction
                }
            ]

                
            
            plan = structured_planner.invoke(messages)
            
            state["plan"] = plan
            print("Plan produced")

        except Exception as e:
            state["error"] = f"Planner failed: {str(e)}"

        return state

    def executor_node(self,state: DataAnalystState) -> DataAnalystState:

        llm = self._llm
        if state.get("error"):
            return state

        try:
            plan = state["plan"]
            executor_system_prompt = self._executor_prompt_builder(self._idiom, 
                                                                   self._idiom_rules, 
                                                                   plan)
            
            executor_agent = create_agent(
                model=llm,
                system_prompt=executor_system_prompt,
                tools=self._tools,
                #response_format=AgentTableResponse,
                #checkpointer= InMemorySaver() 
            )     
            
            
            result = executor_agent.invoke({
                "messages": [
                    {
                        "role": "system",
                        "content": executor_system_prompt
                    }

                ]
            })

            state["execution_result"] = result

        except Exception as e:
            state["execution_result"] = None 
            state["error"] = f"Executor failed: {str(e)}"

        return state







        return state 

    def _build_graph(self):

        builder = StateGraph(DataAnalystState)

        builder.add_node("planner", self.planner_node)
        builder.add_node("executor", self.executor_node)

        builder.set_entry_point("planner")

        builder.add_conditional_edges(
            "planner",
            self._should_continue,
            {
                "executor": "executor",
                "end": END,
            }
        )

        builder.add_edge("executor", END)

        return builder.compile()

    def _should_continue(
        self,
        state: DataAnalystState
    ) -> str:

        if state.get("error"):
            return "end"

        return "executor"
    
    def _planner_prompt_builder( self )->str:
        txt = self._data.catalog_snapshot()
        txt = json.dumps( self._data.catalog_snapshot(), indent=3)

        prompt = system_prompt_sql_planner_template.format(catalog=txt)

        return prompt 

    def _executor_prompt_builder( self, idiom, idiom_examples, plan )->str:
        prompt = system_prompt_sql_executor_template.format(idiom=idiom, idiom_examples=idiom_examples, plan=plan)
        return prompt 



        





: 

: 

: 

: 

In [ ]:
analyst.run( query9 )


: 

In [ ]:
display(analyst._graph )


: 

In [ ]:
for n,item in enumerate(queries):

    print("\n\n")
    print(120*'=')
    user_query = item[0]
    print( user_query, 30*' ', n  )
    print(120*'=')
    

    #user_query = "name the first tree wells in the last table"
    messages = {"messages": [{"role": "user", "content": user_query}]}
    
    try:
        response = run_agent_stream_values(agent, messages ) 
        last_response = response 
    except Exception as e:
        print("STREAM FAILED:", str(e))
    #response = agent.invoke(
    #messages,
    #config={"recursion_limit": 10},
    #)
    print(50*'=',sep="\n\n")
    break

: 

: 